# UTS Data Acquisition & Management
## Akuisisi dan Integrasi Data Pariwisata Mancanegara di Indonesia
**Studi berdasarkan statistik BPS, API Open-Meteo, dan Web Scraping Wikipedia**

| | |
|---|---|
| Mata kuliah | Data Acquisition & Management (Kelas E, Semester Ganjil) |
| Dosen | Dr. I Gede Totok Suryawan, S.Kom., M.T. |
| Program Studi | Informatika, INSTIKI |
| Platform | Google Colab |
| Topik | Pariwisata (wisatawan mancanegara) seluruh Indonesia |

> **Cara menjalankan:** isi sel **0.2 Identitas** (nama, NIM, email kampus), lalu pilih *Runtime → Run all*. Saat sel 0.7 meminta, unggah 3 berkas CSV BPS. Pastikan semua sel sudah dijalankan sebelum notebook dikumpulkan.

## BAGIAN AWAL: Identitas dan Pendahuluan
### 0.1 Judul proyek
**Akuisisi dan Integrasi Data Pariwisata Mancanegara di Indonesia**

### 0.2 Identitas mahasiswa
Isi data di bawah ini. Email kampus dipakai pada header `User-Agent` setiap request web (wajib menurut petunjuk soal).

In [49]:
# ===== 0.2 IDENTITAS (WAJIB DIISI SEBELUM RUN) =====
NAMA_MAHASISWA = "PUTU INTAN CAHYANTI PUTRI"          # contoh: "I Made Contoh"
NIM            = "2501010046"                    # contoh: "2301010001"
KELAS          = "E"
EMAIL_KAMPUS   = "mahasiswa@instiki.ac.id"           # contoh: "nim@mahasiswa.instiki.ac.id"

# Tanggal Anda mengunduh 3 berkas BPS (dipakai di catatan provenance), format YYYY-MM-DD
TANGGAL_UNDUH_BPS = "2026-11-19"

# Hentikan eksekusi bila identitas belum diisi, supaya User-Agent tidak berisi placeholder
if "@" not in EMAIL_KAMPUS or "ISI_" in (NAMA_MAHASISWA + NIM + EMAIL_KAMPUS):
    raise ValueError("Isi NAMA_MAHASISWA, NIM, dan EMAIL_KAMPUS terlebih dahulu.")
print("Identitas terisi:", NAMA_MAHASISWA, "|", NIM, "|", EMAIL_KAMPUS)

Identitas terisi: PUTU INTAN CAHYANTI PUTRI | 2501010046 | mahasiswa@instiki.ac.id


### 0.3 Latar belakang
Indonesia adalah tujuan wisata yang bergantung pada arus wisatawan mancanegara (wisman). Pola kedatangan wisman dipengaruhi banyak faktor: negara asal, pintu masuk, musim, dan cuaca di destinasi. Data statistik resmi (BPS) umumnya terpisah dari data cuaca dan informasi destinasi, sehingga perlu proses **akuisisi, dokumentasi asal-usul (provenance), pemeriksaan etika, dan integrasi** agar dapat dianalisis bersama.

### 0.4 Rumusan masalah
1. Bagaimana membaca dan mengidentifikasi dataset BPS wisman 2025 secara tepat, serta menghitung kebutuhan penyimpanannya?
2. Apakah sumber data (BPS, Open-Meteo, Wikipedia) boleh digunakan, dan adakah risiko PII / quasi-identifier?
3. Bagaimana mengakuisisi data cuaca melalui API dan data halaman statis melalui scraping secara etis?
4. Bagaimana menggabungkan sumber-sumber tersebut secara valid tanpa duplikasi baris?

### 0.5 Tujuan proyek
Membangun alur akuisisi data yang **terdokumentasi, etis, dan dapat direproduksi** dari tiga cara akuisisi (unduh berkas, API, web scraping), lalu mengintegrasikannya menjadi dataset terpadu beserta metadata, provenance, dan Data Ethics Checklist.

### 0.6 Deskripsi sumber data
| ID | Sumber | Data | Metode akuisisi |
|---|---|---|---|
| BPS-01 | BPS | Kunjungan wisman per bulan menurut paspor yang dipegang, 2025 | Unduh berkas CSV |
| BPS-02 | BPS | Rata-rata pengeluaran wisman per kunjungan menurut kebangsaan (US$), 2025 | Unduh berkas CSV |
| BPS-03 | BPS | Kunjungan wisman per bulan menurut pintu masuk, 2025 | Unduh berkas CSV |
| API-01 | Open-Meteo (Historical Weather API) | Cuaca harian 2025 di lokasi pintu masuk utama | API (JSON) |
| WEB-01 | Wikipedia | Halaman "Pariwisata di Indonesia" | Web scraping (HTML statis) |

Tautan portal BPS: https://www.bps.go.id/id/statistics-table?subject=561

### 0.7 Library dan environment Python

In [50]:
# ===== 0.7 IMPORT LIBRARY, KONFIGURASI, DAN STRUKTUR FOLDER =====
import os, re, io, json, time, hashlib, shutil, sys, platform
from pathlib import Path
from datetime import datetime, timezone, timedelta
from urllib.robotparser import RobotFileParser

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

try:
    display            # tersedia di Colab/Jupyter
except NameError:
    display = print

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Zona waktu WIB untuk pencatatan tanggal akses
WIB = timezone(timedelta(hours=7))
def now_wib():
    return datetime.now(WIB).strftime("%Y-%m-%d %H:%M:%S WIB")

# Struktur folder akhir proyek


BASE_DIR = Path("UTS_DAM_Pariwisata_Indonesia")
RAW  = BASE_DIR / "data" / "raw"          # data mentah (tidak boleh diubah)
PROC = BASE_DIR / "data" / "processed"    # data hasil pembersihan
DOC  = BASE_DIR / "documentation"         # provenance, checklist, bukti
for d in (RAW, PROC, DOC, DOC / "bukti_lisensi"):
    d.mkdir(parents=True, exist_ok=True)

# .gitignore: pastikan berkas rahasia tidak ikut terunggah ke GitHub
(BASE_DIR / ".gitignore").write_text(".env\n*.key\ncredentials*.json\n__pycache__/\n", encoding="utf-8")

print("Python :", sys.version.split()[0], "|", platform.platform())
print("pandas :", pd.__version__, "| requests:", requests.__version__)
print("Folder proyek:", BASE_DIR.resolve())

Python : 3.13.18 | Linux-6.6.159+-x86_64-with-glibc2.39
pandas : 2.2.3 | requests: 2.32.4
Folder proyek: /content/UTS_DAM_Pariwisata_Indonesia


In [51]:
# ===== 0.7b KONFIGURASI REQUEST WEB (timeout, User-Agent, jeda, retry) =====
# Semua request ke web WAJIB lewat fungsi http_get() di bawah ini.
USER_AGENT = (f"UTS-DAM-INSTIKI/1.0 (tugas kuliah; {NAMA_MAHASISWA}; NIM {NIM}; "
              f"kontak: {EMAIL_KAMPUS}) python-requests/{requests.__version__}")
TIMEOUT = (5, 30)          # (connect timeout, read timeout) dalam detik
JEDA_DETIK = 2.0           # jeda setelah setiap request agar server tidak terbebani
MAX_RETRY = 4              # maksimum percobaan per URL
RETRY_STATUS = {429, 500, 502, 503, 504}   # status sementara -> boleh dicoba ulang

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})

REQUEST_LOG = []           # bukti setiap percobaan request (status, durasi, dst.)

def http_get(url, params=None, max_retry=MAX_RETRY, jeda=None):
    """GET dengan timeout, User-Agent, penanganan status code, retry (exponential
    backoff), dan jeda antar-request. Mengembalikan objek Response berstatus 200."""
    jeda = JEDA_DETIK if jeda is None else jeda
    for percobaan in range(1, max_retry + 1):
        t0 = time.time()
        resp, status, galat = None, None, ""
        try:
            resp = SESSION.get(url, params=params, timeout=TIMEOUT)
            status = resp.status_code
        except (requests.Timeout, requests.ConnectionError) as e:
            galat = type(e).__name__             # timeout / koneksi putus -> retry
        REQUEST_LOG.append({
            "waktu": now_wib(), "url": url, "percobaan": percobaan,
            "status": status, "galat": galat, "durasi_detik": round(time.time() - t0, 2)})

        if status == 200:                         # sukses
            time.sleep(jeda)                      # jeda antar-request
            return resp
        if status is not None and status not in RETRY_STATUS:
            time.sleep(jeda)
            resp.raise_for_status()               # 4xx lain (400/403/404): jangan retry
            return resp                           # 2xx/3xx selain 200: kembalikan apa adanya
        # Status sementara (429/5xx) atau galat jaringan -> tunggu lalu ulangi
        tunggu = jeda * (2 ** (percobaan - 1))    # backoff: 2, 4, 8, 16 detik
        if resp is not None and resp.headers.get("Retry-After", "").isdigit():
            tunggu = max(tunggu, int(resp.headers["Retry-After"]))
        print(f"  [retry] {url} status={status} {galat} (percobaan {percobaan}/{max_retry}), tunggu {tunggu}s")
        time.sleep(tunggu)
    raise RuntimeError(f"Gagal mengambil {url} setelah {max_retry} percobaan")

print("User-Agent :", USER_AGENT)
print("Timeout    :", TIMEOUT, "| Jeda:", JEDA_DETIK, "dtk | Max retry:", MAX_RETRY)

User-Agent : UTS-DAM-INSTIKI/1.0 (tugas kuliah; PUTU INTAN CAHYANTI PUTRI; NIM 2501010046; kontak: mahasiswa@instiki.ac.id) python-requests/2.32.4
Timeout    : (5, 30) | Jeda: 2.0 dtk | Max retry: 4


In [52]:
# ===== 0.7c UNGGAH BERKAS BPS KE data/raw (DISALIN APA ADANYA, TIDAK DIUBAH) =====
# Nama berkas asli hasil unduhan BPS  ->  nama standar di folder data/raw
BERKAS_BPS = {
    "bps_paspor_2025.csv":
        "Jumlah_Kunjungan_Wisatawan_Mancanegara_per_bulan_Menurut_Paspor_yang_Dipegang___2025.csv",
    "bps_pengeluaran_2025.csv":
        "Rata-Rata_Pengeluaran_Wisatawan_Mancanegara_per_Kunjungan_Menurut_Kebangsaan__2025.csv",
    "bps_pintu_masuk_2025.csv":
        "Jumlah_Kunjungan_Wisatawan_Mancanegara_per_bulan_ke_Indonesia_Menurut_Pintu_Masuk__2008_-_sekarang__2025.csv",
}

def normalisasi_nama_file(nama):
    """Menormalisasi nama file dengan menghilangkan spasi, underscore, dan karakter non-alfanumerik agar pencocokan fleksibel."""
    return re.sub(r"[^a-zA-Z0-9]", "", nama).lower()

def cari_berkas_fleksibel(nama_target):
    """Cari berkas di folder kerja Colab secara fleksibel dengan membandingkan nama ter-normalisasi."""
    target_norm = normalisasi_nama_file(nama_target)
    for folder in (Path("."), Path("/content"), Path("/mnt/user-data/uploads")):
        if folder.exists():
            for item in folder.iterdir():
                if item.is_file() and normalisasi_nama_file(item.name) == target_norm:
                    return item
    return None

# Bila ada berkas yang belum ditemukan, minta unggah (khusus Google Colab)
if any(cari_berkas_fleksibel(n) is None for n in BERKAS_BPS.values()):
    from google.colab import files
    print("Unggah ketiga berkas CSV BPS:")
    files.upload()

ORIGINAL_MD5 = {}
for nama_std, nama_asli in BERKAS_BPS.items():
    sumber = cari_berkas_fleksibel(nama_asli)
    assert sumber is not None, f"Berkas tidak ditemukan: {nama_asli} (Cek kembali nama file yang diunggah)"
    shutil.copyfile(sumber, RAW / nama_std)        # salin byte-per-byte
    ORIGINAL_MD5[nama_std] = hashlib.md5(Path(sumber).read_bytes()).hexdigest()
    print(f"OK  {sumber.name[:60]}...  ->  data/raw/{nama_std}")

PATH_PASPOR     = RAW / "bps_paspor_2025.csv"
PATH_PENGELUARAN = RAW / "bps_pengeluaran_2025.csv"
PATH_PINTU      = RAW / "bps_pintu_masuk_2025.csv"

OK  Jumlah Kunjungan Wisatawan Mancanegara per bulan Menurut Pas...  ->  data/raw/bps_paspor_2025.csv
OK  Rata-Rata Pengeluaran Wisatawan Mancanegara per Kunjungan Me...  ->  data/raw/bps_pengeluaran_2025.csv
OK  Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesi...  ->  data/raw/bps_pintu_masuk_2025.csv


In [53]:
# ===== FUNGSI PROVENANCE (dipakai setiap kali satu sumber berhasil diakuisisi) =====
PROVENANCE = []

def md5_file(path):
    """Hash MD5 isi berkas (sidik jari integritas)."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(8192), b""):
            h.update(blok)
    return h.hexdigest()

def catat_provenance(source_id, nama_sumber, nama_data, url, metode, path,
                     lisensi, format_data, tanggal_akses=None, catatan=""):
    PROVENANCE.append({
        "source_id": source_id, "nama_sumber": nama_sumber, "nama_data": nama_data,
        "url": url, "metode_akuisisi": metode,
        "tanggal_akses": tanggal_akses or now_wib(),
        "lisensi": lisensi, "format_data": format_data,
        "file_mentah": str(path), "ukuran_byte": Path(path).stat().st_size,
        "md5": md5_file(path), "catatan": catatan})

BPS_URL = "https://www.bps.go.id/id/statistics-table?subject=561"
BPS_LISENSI = ("Hak cipta BPS; boleh dikutip/digunakan dengan memberi kredit kepada BPS "
               "(judul konten, tanggal akses, tautan asli) - Ketentuan Penggunaan BPS")

for sid, (nama_std, judul) in zip(
        ["BPS-01", "BPS-02", "BPS-03"],
        [("bps_paspor_2025.csv", "Jumlah Kunjungan Wisatawan Mancanegara per Bulan Menurut Paspor yang Dipegang, 2025"),
         ("bps_pengeluaran_2025.csv", "Rata-Rata Pengeluaran Wisatawan Mancanegara per Kunjungan Menurut Kebangsaan, 2025"),
         ("bps_pintu_masuk_2025.csv", "Jumlah Kunjungan Wisatawan Mancanegara per Bulan ke Indonesia Menurut Pintu Masuk, 2008-sekarang (data 2025)")]):
    catat_provenance(sid, "Badan Pusat Statistik (BPS)", judul, BPS_URL, "Unduh berkas CSV",
                     RAW / nama_std, BPS_LISENSI, "CSV",
                     tanggal_akses=TANGGAL_UNDUH_BPS,
                     catatan="MD5 salinan di data/raw sama dengan berkas hasil unduhan: "
                             + str(md5_file(RAW / nama_std) == ORIGINAL_MD5[nama_std]))
display(pd.DataFrame(PROVENANCE)[["source_id", "nama_data", "tanggal_akses", "md5", "catatan"]])

,source_id,nama_data,tanggal_akses,md5,catatan
0,BPS-01,"Jumlah Kunjungan Wisatawan Mancanegara per Bulan Menurut Paspor yang Dipegang, 2025",2026-11-19,f1b7a01180045f68bb3bf56912f887a1,MD5 salinan di data/raw sama dengan berkas hasil unduhan: True
1,BPS-02,"Rata-Rata Pengeluaran Wisatawan Mancanegara per Kunjungan Menurut Kebangsaan, 2025",2026-11-19,13011039e4b605478e1fc65673c2b1a9,MD5 salinan di data/raw sama dengan berkas hasil unduhan: True
2,BPS-03,"Jumlah Kunjungan Wisatawan Mancanegara per Bulan ke Indonesia Menurut Pintu Masuk, 2008-sekarang (data 2025)",2026-11-19,0a25d32a1fc15e65974727e20f22a783,MD5 salinan di data/raw sama dengan berkas hasil unduhan: True


---
# SOAL 1 · Pembacaan dan Identifikasi Dataset (bobot 20%)
## BAGIAN 1: Pembacaan dan Identifikasi Dataset
### 1.1 Identifikasi sumber dataset BPS
Ketiga berkas diunduh dari **BPS Statistik Pariwisata** (https://www.bps.go.id/id/statistics-table?subject=561). Berkas BPS memiliki **baris judul di atas tabel** (nama variabel, judul, tahun), kolom pertama tanpa nama, dan angka tanpa pemisah ribuan. Karena itu parameter baca yang dipakai: `skiprows`, `encoding="utf-8-sig"` (berkas berawalan BOM), `na_values`, dan `dtype`.

### 1.2 Pembacaan berkas CSV dengan parameter yang tepat
Pertama, lihat struktur mentah apa adanya (`header=None`) untuk membuktikan alasan pemilihan `skiprows`.

In [54]:
# Lihat 6 baris pertama berkas mentah apa adanya untuk menentukan skiprows/header
for nama in ["bps_paspor_2025.csv", "bps_pengeluaran_2025.csv", "bps_pintu_masuk_2025.csv"]:
    print("=" * 100, "\n", nama)
    display(pd.read_csv(RAW / nama, header=None, nrows=6, encoding="utf-8-sig", dtype=str))

 bps_paspor_2025.csv


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,Paspor yang dipegang,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Jumlah Kunjungan Wisatawan Mancanegara per bulan Menurut Paspor yang Dipegang (Kunjungan),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
4,Malaysia,221474,214605,159895,170018,238497,236417,212113,229726,272421,234384,207037,243162,2639749
5,Singapore,110484,89757,100883,97506,126391,183748,114224,129864,119237,123578,126254,204512,1526438


 bps_pengeluaran_2025.csv


,0,1
0,Negara Tempat Tinggal,NaN
1,NaN,Rata-Rata Pengeluaran Wisatawan Mancanegara per Kunjungan Menurut Kebangsaan (US $)
2,NaN,2025
3,Brunei Darussalam,1157
4,Malaysia,594
5,Filipina,1002


 bps_pintu_masuk_2025.csv


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,Pintu Masuk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang (Kunjungan)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
4,A. Pintu Udara,801614,718609,669479,854756,941092,978929,1101619,1108133,996518,954216,805662,888664,10819291
5,Ngurah Rai,526831,444411,464697,587315,602055,637744,697068,682839,634482,594846,476507,558790,6907585


In [55]:
# ===== 1.2 PEMBACAAN DENGAN PARAMETER YANG TEPAT =====
BULAN = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli",
         "Agustus", "September", "Oktober", "November", "Desember"]
DTYPE_BULAN = {b: "int64" for b in BULAN + ["Tahunan"]}   # kolom angka = bilangan bulat

# (1) Kunjungan menurut paspor: 3 baris judul dilewati, baris ke-4 = header
df_paspor_raw = pd.read_csv(
    PATH_PASPOR,
    sep=",",                 # pemisah kolom
    encoding="utf-8-sig",    # menghapus BOM di awal berkas
    skiprows=3,              # lewati 3 baris judul (variabel, judul tabel, tahun)
    na_values=["-", "", " "],# penanda data kosong BPS
    dtype=DTYPE_BULAN,       # paksa tipe angka
)
df_paspor_raw = df_paspor_raw.rename(columns={df_paspor_raw.columns[0]: "paspor"})
df_paspor_raw["paspor"] = df_paspor_raw["paspor"].astype(str).str.strip()

# (2) Rata-rata pengeluaran: tidak punya baris header sendiri -> header=None + names
df_pengeluaran_raw = pd.read_csv(
    PATH_PENGELUARAN,
    encoding="utf-8-sig", skiprows=3, header=None,
    names=["negara", "pengeluaran_usd"],
    na_values=["-", "", " "],
    dtype={"negara": "string", "pengeluaran_usd": "int64"},
)
df_pengeluaran_raw["negara"] = df_pengeluaran_raw["negara"].str.strip()

# (3) Kunjungan menurut pintu masuk
df_pintu_raw = pd.read_csv(
    PATH_PINTU,
    encoding="utf-8-sig", skiprows=3, na_values=["-", "", " "], dtype=DTYPE_BULAN,
)
df_pintu_raw = df_pintu_raw.rename(columns={df_pintu_raw.columns[0]: "pintu_masuk"})
df_pintu_raw["pintu_masuk"] = df_pintu_raw["pintu_masuk"].astype(str).str.strip()

DATASETS = {"BPS-01 Paspor": df_paspor_raw,
            "BPS-02 Pengeluaran": df_pengeluaran_raw,
            "BPS-03 Pintu Masuk": df_pintu_raw}
for k, v in DATASETS.items():
    print("=" * 100, "\n", k, v.shape)
    display(v.head(5))

 BPS-01 Paspor (248, 14)


,paspor,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
0,Malaysia,221474,214605,159895,170018,238497,236417,212113,229726,272421,234384,207037,243162,2639749
1,Singapore,110484,89757,100883,97506,126391,183748,114224,129864,119237,123578,126254,204512,1526438
2,Timor Leste,75735,61770,63471,69721,80809,82296,90840,89516,86861,90077,98987,119098,1009181
3,Indonesia,46451,42534,35310,45118,51180,69252,55391,62962,54550,58476,50980,59812,632016
4,Philippines,16370,15505,15457,18652,23264,24580,22132,22278,20437,22948,22054,23051,246728


 BPS-02 Pengeluaran (37, 2)


,negara,pengeluaran_usd
0,Brunei Darussalam,1157
1,Malaysia,594
2,Filipina,1002
3,Singapura,624
4,Thailand,1034


 BPS-03 Pintu Masuk (35, 14)


,pintu_masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
0,A. Pintu Udara,801614,718609,669479,854756,941092,978929,1101619,1108133,996518,954216,805662,888664,10819291
1,Ngurah Rai,526831,444411,464697,587315,602055,637744,697068,682839,634482,594846,476507,558790,6907585
2,Soekarno-Hatta,193433,199306,146390,186523,233751,236013,290369,304970,244404,254968,240330,230381,2760838
3,Juanda,22119,19205,15647,24800,33409,34092,34421,37406,32664,29534,21629,25019,329945
4,Kualanamu,23011,22546,16881,19709,26183,25183,28306,29858,29107,24621,20826,23425,289656


### 1.3 Pemeriksaan struktur dataset

In [56]:
for k, v in DATASETS.items():
    print("=" * 100, "\n", k)
    v.info()
    print("Total nilai kosong:", int(v.isna().sum().sum()),
          "| Baris duplikat penuh:", int(v.duplicated().sum()))

 BPS-01 Paspor
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248 entries, 0 to 247
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   paspor     248 non-null    object
 1   Januari    248 non-null    int64 
 2   Februari   248 non-null    int64 
 3   Maret      248 non-null    int64 
 4   April      248 non-null    int64 
 5   Mei        248 non-null    int64 
 6   Juni       248 non-null    int64 
 7   Juli       248 non-null    int64 
 8   Agustus    248 non-null    int64 
 9   September  248 non-null    int64 
 10  Oktober    248 non-null    int64 
 11  November   248 non-null    int64 
 12  Desember   248 non-null    int64 
 13  Tahunan    248 non-null    int64 
dtypes: int64(13), object(1)
memory usage: 27.3+ KB
Total nilai kosong: 0 | Baris duplikat penuh: 0
 BPS-02 Pengeluaran
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 2 columns):
 #   Column           Non-Null Cou

### 1.4 Jenis data
Ketiga dataset adalah **data terstruktur**: berbentuk tabel baris-kolom dalam format CSV dengan skema tetap. Data berformat wide (satu kolom per bulan) dan memuat baris subtotal/agregat BPS (mis. `TOTAL ASIA`, `A. Pintu Udara`) yang harus dibedakan dari baris rinci saat analisis. Hasil API (JSON) bersifat **semi-terstruktur**, dan halaman Wikipedia (HTML) **semi-terstruktur** sebelum diekstrak.

### 1.5 Jumlah baris dan kolom

In [57]:
ringkas = pd.DataFrame([{"dataset": k, "jumlah_baris": v.shape[0], "jumlah_kolom": v.shape[1],
                         "memenuhi_min_100_baris": v.shape[0] >= 100} for k, v in DATASETS.items()])
display(ringkas)
print("\nCatatan: syarat soal (dataset terbuka minimal 100 baris) dipenuhi oleh BPS-01 (paspor).")
print("BPS-02 dan BPS-03 berukuran kecil sehingga berperan sebagai data pendukung.")

,dataset,jumlah_baris,jumlah_kolom,memenuhi_min_100_baris
0,BPS-01 Paspor,248,14,True
1,BPS-02 Pengeluaran,37,2,False
2,BPS-03 Pintu Masuk,35,14,False



Catatan: syarat soal (dataset terbuka minimal 100 baris) dipenuhi oleh BPS-01 (paspor).
BPS-02 dan BPS-03 berukuran kecil sehingga berperan sebagai data pendukung.


### 1.6 Identifikasi tipe data setiap kolom

In [58]:
for k, v in DATASETS.items():
    print("=" * 100, "\n", k)
    display(pd.DataFrame({"kolom": v.columns, "dtype": v.dtypes.astype(str).values,
                          "contoh_nilai": [v[c].iloc[0] for c in v.columns]}))

 BPS-01 Paspor


,kolom,dtype,contoh_nilai
0,paspor,object,Malaysia
1,Januari,int64,221474
2,Februari,int64,214605
3,Maret,int64,159895
4,April,int64,170018
5,Mei,int64,238497
6,Juni,int64,236417
7,Juli,int64,212113
8,Agustus,int64,229726
9,September,int64,272421


 BPS-02 Pengeluaran


,kolom,dtype,contoh_nilai
0,negara,string,Brunei Darussalam
1,pengeluaran_usd,int64,1157


 BPS-03 Pintu Masuk


,kolom,dtype,contoh_nilai
0,pintu_masuk,object,A. Pintu Udara
1,Januari,int64,801614
2,Februari,int64,718609
3,Maret,int64,669479
4,April,int64,854756
5,Mei,int64,941092
6,Juni,int64,978929
7,Juli,int64,1101619
8,Agustus,int64,1108133
9,September,int64,996518


### 1.7 Identifikasi satuan data

In [59]:
SATUAN = pd.DataFrame([
    {"dataset": "BPS-01 Paspor", "variabel": "Januari-Desember, Tahunan", "satuan": "Kunjungan (jumlah kedatangan wisman)", "keterangan": "Per bulan; kolom Tahunan = jumlah 12 bulan"},
    {"dataset": "BPS-02 Pengeluaran", "variabel": "pengeluaran_usd", "satuan": "US$ per kunjungan", "keterangan": "Rata-rata pengeluaran wisman per kunjungan menurut kebangsaan"},
    {"dataset": "BPS-03 Pintu Masuk", "variabel": "Januari-Desember, Tahunan", "satuan": "Kunjungan (jumlah kedatangan wisman)", "keterangan": "Dikelompokkan pintu udara, laut, darat"},
    {"dataset": "API-01 Open-Meteo", "variabel": "suhu / hujan", "satuan": "derajat C / mm (lihat daily_units di respons)", "keterangan": "Data harian"},
])
display(SATUAN)

,dataset,variabel,satuan,keterangan
0,BPS-01 Paspor,"Januari-Desember, Tahunan",Kunjungan (jumlah kedatangan wisman),Per bulan; kolom Tahunan = jumlah 12 bulan
1,BPS-02 Pengeluaran,pengeluaran_usd,US$ per kunjungan,Rata-rata pengeluaran wisman per kunjungan menurut kebangsaan
2,BPS-03 Pintu Masuk,"Januari-Desember, Tahunan",Kunjungan (jumlah kedatangan wisman),"Dikelompokkan pintu udara, laut, darat"
3,API-01 Open-Meteo,suhu / hujan,derajat C / mm (lihat daily_units di respons),Data harian


### 1.8 Cakupan wilayah dan waktu

In [60]:
n_negara = (~df_paspor_raw["paspor"].str.contains("TOTAL|A S E A N", regex=True)).sum()
cakupan = pd.DataFrame([
    {"dataset": "BPS-01 Paspor", "wilayah": f"Seluruh Indonesia (kedatangan wisman); {n_negara} baris negara/kelompok paspor", "waktu": "Januari-Desember 2025 (bulanan)"},
    {"dataset": "BPS-02 Pengeluaran", "wilayah": f"Seluruh Indonesia; {len(df_pengeluaran_raw)-2} negara/kelompok negara tempat tinggal", "waktu": "2025 (tahunan)"},
    {"dataset": "BPS-03 Pintu Masuk", "wilayah": "Seluruh Indonesia; pintu udara, laut, darat", "waktu": "Januari-Desember 2025 (bulanan); tabel BPS aslinya memuat seri 2008-sekarang, berkas ini memuat 2025"},
])
display(cakupan)

,dataset,wilayah,waktu
0,BPS-01 Paspor,Seluruh Indonesia (kedatangan wisman); 240 baris negara/kelompok paspor,Januari-Desember 2025 (bulanan)
1,BPS-02 Pengeluaran,Seluruh Indonesia; 35 negara/kelompok negara tempat tinggal,2025 (tahunan)
2,BPS-03 Pintu Masuk,"Seluruh Indonesia; pintu udara, laut, darat","Januari-Desember 2025 (bulanan); tabel BPS aslinya memuat seri 2008-sekarang, berkas ini memuat 2025"


### 1.9 Pemeriksaan ukuran masing-masing file

In [61]:
def ukuran_tabel(file_map):
    rows = []
    for nama, p in file_map.items():
        b = Path(p).stat().st_size
        rows.append({"berkas": nama, "ukuran_byte": b, "ukuran_kb": round(b / 1024, 3)})
    return pd.DataFrame(rows)

display(ukuran_tabel({"bps_paspor_2025.csv": PATH_PASPOR,
                      "bps_pengeluaran_2025.csv": PATH_PENGELUARAN,
                      "bps_pintu_masuk_2025.csv": PATH_PINTU}))

,berkas,ukuran_byte,ukuran_kb
0,bps_paspor_2025.csv,14815,14.468
1,bps_pengeluaran_2025.csv,673,0.657
2,bps_pintu_masuk_2025.csv,3188,3.113


### 1.10 Estimasi kebutuhan storage selama satu tahun
Rumus dari soal:

**Storage = (Ukuran berkas × 12 bulan) × 1,3** (cadangan 30%)

Fungsi di bawah dipakai dua kali: sekarang untuk 3 berkas BPS, dan nanti (bagian 6) ditambah hasil API dan scraping.

In [62]:
CADANGAN = 0.30     # cadangan 30% sesuai soal
BULAN_ARSIP = 12    # 12 bulan

def hitung_storage(file_map):
    """Storage 1 tahun = (ukuran x 12) x (1 + 30%)."""
    rows = []
    for nama, p in file_map.items():
        kb = Path(p).stat().st_size / 1024
        total_12 = kb * BULAN_ARSIP
        cad = total_12 * CADANGAN
        rows.append({"berkas": nama, "ukuran_kb": round(kb, 3),
                     "x12_bulan_kb": round(total_12, 3),
                     "cadangan_30persen_kb": round(cad, 3),
                     "storage_total_kb": round(total_12 + cad, 3)})
    df = pd.DataFrame(rows)
    total = {"berkas": "TOTAL"}
    for c in df.columns[1:]:
        total[c] = round(df[c].sum(), 3)
    df = pd.concat([df, pd.DataFrame([total])], ignore_index=True)
    df["storage_total_mb"] = (df["storage_total_kb"] / 1024).round(4)
    return df

FILES_STORAGE = {"bps_paspor_2025.csv": PATH_PASPOR,
                 "bps_pengeluaran_2025.csv": PATH_PENGELUARAN,
                 "bps_pintu_masuk_2025.csv": PATH_PINTU}
df_storage = hitung_storage(FILES_STORAGE)
display(df_storage)
df_storage.to_csv(DOC / "storage_estimation.csv", index=False)

# Contoh verifikasi rumus dengan angka dari konsep: 2 MB -> 31,2 MB
assert abs((2 * 12) * 1.3 - 31.2) < 1e-9
print("Verifikasi rumus: (2 MB x 12) x 1,3 = 31,2 MB  [OK]")

,berkas,ukuran_kb,x12_bulan_kb,cadangan_30persen_kb,storage_total_kb,storage_total_mb
0,bps_paspor_2025.csv,14.468,173.613,52.084,225.697,0.2204
1,bps_pengeluaran_2025.csv,0.657,7.887,2.366,10.253,0.0100
2,bps_pintu_masuk_2025.csv,3.113,37.359,11.208,48.567,0.0474
3,TOTAL,18.238,218.859,65.658,284.517,0.2778


Verifikasi rumus: (2 MB x 12) x 1,3 = 31,2 MB  [OK]


### 1.11 Penjelasan asumsi perhitungan storage
1. Berkas BPS yang dipakai adalah **data tahun 2025 yang sudah lengkap** (12 bulan dalam satu berkas). Estimasi satu tahun mengikuti rumus soal dengan asumsi **skenario pengarsipan bulanan**: setiap bulan pipeline mengakuisisi dan menyimpan **satu salinan (snapshot) baru** dengan ukuran yang sama dengan berkas saat ini.
2. Ukuran berkas dianggap **tetap** selama 12 bulan (tidak ada pertumbuhan jumlah baris/kolom).
3. **Cadangan 30%** mencakup versi data olahan, log, dan pertumbuhan data.
4. Yang dihitung hanya berkas data; tidak mencakup notebook, dokumentasi, atau biaya cadangan di luar rumus soal.
5. Hasil API dan HTML Wikipedia ditambahkan ke estimasi pada bagian 6.1 setelah berkasnya ada.

---
# SOAL 2 · Etika Data dan Lisensi (bobot 20% bersama Soal 4)
## BAGIAN 2: Data Ethics dan Lisensi
### 2.1 - 2.3 Pemeriksaan lisensi BPS, Open-Meteo, dan Wikipedia
Notebook mengambil halaman ketentuan/lisensi tiap sumber (dengan timeout, User-Agent, dan jeda), menyimpannya sebagai **bukti**, lalu mencari kata kunci lisensi secara otomatis. Teks halaman disimpan di `documentation/bukti_lisensi/`.

In [63]:
# ===== 2.1-2.3 BUKTI LISENSI DAN KETENTUAN PORTAL =====
HALAMAN_LISENSI = [
    # (kode, sumber, url halaman ketentuan/lisensi)
    ("BPS",        "BPS",        "https://www.bps.go.id/assets/term-of-use"),
    ("OPENMETEO",  "Open-Meteo", "https://open-meteo.com/en/license"),
    ("OPENMETEO_TERMS", "Open-Meteo", "https://open-meteo.com/en/terms"),
    ("WIKIPEDIA",  "Wikipedia",  "https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use"),
]
KATA_KUNCI = r"(CC BY[- ]SA|CC BY|Creative Commons|hak cipta|copyright|kutipan|kredit|attribution|non-?commercial|license|lisensi)"

bukti_lisensi = []
for kode, sumber, url in HALAMAN_LISENSI:
    try:
        r = http_get(url)
        teks = BeautifulSoup(r.text, "html.parser").get_text(" ", strip=True)
        (DOC / "bukti_lisensi" / f"{kode}.txt").write_text(
            f"URL: {url}\nTanggal akses: {now_wib()}\nStatus: {r.status_code}\n\n{teks[:20000]}",
            encoding="utf-8")
        ketemu = sorted({m.group(0).lower() for m in re.finditer(KATA_KUNCI, teks, flags=re.I)})
        cuplikan = ""
        m = re.search(KATA_KUNCI, teks, flags=re.I)
        if m:
            cuplikan = teks[max(0, m.start() - 80): m.end() + 120]
        bukti_lisensi.append({"kode": kode, "sumber": sumber, "url": url, "status": r.status_code,
                              "tanggal_akses": now_wib(), "kata_kunci_ditemukan": ", ".join(ketemu),
                              "cuplikan": cuplikan})
    except Exception as e:
        bukti_lisensi.append({"kode": kode, "sumber": sumber, "url": url, "status": "dilengkapi",
                              "tanggal_akses": now_wib(), "kata_kunci_ditemukan": "",
                              "cuplikan": f"{type(e).__name__}: {e}"})

df_bukti_lisensi = pd.DataFrame(bukti_lisensi)
display(df_bukti_lisensi[["kode", "url", "status", "kata_kunci_ditemukan"]])
df_bukti_lisensi.to_csv(DOC / "bukti_lisensi" / "ringkasan_bukti_lisensi.csv", index=False)

,kode,url,status,kata_kunci_ditemukan
0,BPS,https://www.bps.go.id/assets/term-of-use,dilengkapi,
1,OPENMETEO,https://open-meteo.com/en/license,200,
2,OPENMETEO_TERMS,https://open-meteo.com/en/terms,200,"copyright, creative commons, non-commercial"
3,WIKIPEDIA,https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use,200,"attribution, cc by-sa, copyright, creative commons, license"


In [64]:
# Ringkasan kesimpulan lisensi (diverifikasi terhadap bukti di atas dan di folder bukti_lisensi)
df_lisensi = pd.DataFrame([
    {"sumber": "BPS", "lisensi_ketentuan": "Materi/data tunduk pada hak cipta BPS; boleh dipakai dengan memberi kutipan/kredit kepada BPS (judul konten, tanggal akses, penulis, tautan asli).",
     "kewajiban": "Cantumkan BPS sebagai sumber + tautan + tanggal akses", "bukti": "documentation/bukti_lisensi/BPS.txt"},
    {"sumber": "Open-Meteo", "lisensi_ketentuan": "Data cuaca CC BY 4.0; API gratis untuk penggunaan non-komersial tanpa API key, dengan batas request harian.",
     "kewajiban": "Atribusi ke Open-Meteo.com; jangan melebihi batas request; hanya non-komersial", "bukti": "documentation/bukti_lisensi/OPENMETEO.txt"},
    {"sumber": "Wikipedia", "lisensi_ketentuan": "Teks tersedia di bawah CC BY-SA 4.0 (Wikimedia Terms of Use).",
     "kewajiban": "Atribusi + bagikan turunan dengan lisensi sama; patuhi robots.txt dan kebijakan User-Agent", "bukti": "documentation/bukti_lisensi/WIKIPEDIA.txt"},
])
pd.set_option("display.max_colwidth", 140)
display(df_lisensi)
print("Penting: pastikan kolom 'kata_kunci_ditemukan' di atas terisi dan baca berkas bukti_lisensi untuk memastikan ringkasan ini sesuai teks resmi.")

,sumber,lisensi_ketentuan,kewajiban,bukti
0,BPS,"Materi/data tunduk pada hak cipta BPS; boleh dipakai dengan memberi kutipan/kredit kepada BPS (judul konten, tanggal akses, penulis, tau...",Cantumkan BPS sebagai sumber + tautan + tanggal akses,documentation/bukti_lisensi/BPS.txt
1,Open-Meteo,"Data cuaca CC BY 4.0; API gratis untuk penggunaan non-komersial tanpa API key, dengan batas request harian.",Atribusi ke Open-Meteo.com; jangan melebihi batas request; hanya non-komersial,documentation/bukti_lisensi/OPENMETEO.txt
2,Wikipedia,Teks tersedia di bawah CC BY-SA 4.0 (Wikimedia Terms of Use).,Atribusi + bagikan turunan dengan lisensi sama; patuhi robots.txt dan kebijakan User-Agent,documentation/bukti_lisensi/WIKIPEDIA.txt


Penting: pastikan kolom 'kata_kunci_ditemukan' di atas terisi dan baca berkas bukti_lisensi untuk memastikan ringkasan ini sesuai teks resmi.


### 2.4 Identifikasi PII
**PII (Personally Identifiable Information)** adalah data yang dapat mengidentifikasi individu (nama orang, email, telepon, NIK, alamat, dsb.). Pemeriksaan dilakukan atas nama kolom dan isi sel ketiga dataset BPS.

In [65]:
# ===== 2.4 PEMERIKSAAN PII =====
POLA_PII = {
    "email":   r"[\w\.-]+@[\w\.-]+\.\w+",
    "telepon": r"(\+62|0)8\d{8,11}",
    "nik_16_digit": r"\b\d{16}\b",
}
NAMA_KOLOM_PII = r"(nama|name|email|telepon|phone|alamat|address|nik|ktp|passport_no|nomor_paspor)"

hasil_pii = []
for k, v in DATASETS.items():
    kolom_mencurigakan = [c for c in v.columns if re.search(NAMA_KOLOM_PII, str(c), flags=re.I)]
    teks = v.astype(str)
    for jenis, pola in POLA_PII.items():
        jumlah = int(teks.apply(lambda s: s.str.contains(pola, regex=True)).sum().sum())
        hasil_pii.append({"dataset": k, "jenis_pemeriksaan": f"pola {jenis}", "temuan": jumlah})
    hasil_pii.append({"dataset": k, "jenis_pemeriksaan": "nama kolom bernuansa PII", "temuan": len(kolom_mencurigakan)})
df_pii = pd.DataFrame(hasil_pii)
display(df_pii)
TOTAL_PII = int(df_pii["temuan"].sum())
print("Total temuan PII:", TOTAL_PII)
print("Kesimpulan:", "tidak ditemukan PII." if TOTAL_PII == 0 else "ada temuan, periksa manual.")

/tmp/ipykernel_2505/542063613.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  jumlah = int(teks.apply(lambda s: s.str.contains(pola, regex=True)).sum().sum())
/tmp/ipykernel_2505/542063613.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  jumlah = int(teks.apply(lambda s: s.str.contains(pola, regex=True)).sum().sum())
/tmp/ipykernel_2505/542063613.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  jumlah = int(teks.apply(lambda s: s.str.contains(pola, regex=True)).sum().sum())


,dataset,jenis_pemeriksaan,temuan
0,BPS-01 Paspor,pola email,0
1,BPS-01 Paspor,pola telepon,0
2,BPS-01 Paspor,pola nik_16_digit,0
3,BPS-01 Paspor,nama kolom bernuansa PII,0
4,BPS-02 Pengeluaran,pola email,0
5,BPS-02 Pengeluaran,pola telepon,0
6,BPS-02 Pengeluaran,pola nik_16_digit,0
7,BPS-02 Pengeluaran,nama kolom bernuansa PII,0
8,BPS-03 Pintu Masuk,pola email,0
9,BPS-03 Pintu Masuk,pola telepon,0


Total temuan PII: 0
Kesimpulan: tidak ditemukan PII.


### 2.5 Identifikasi Quasi-Identifier
**Quasi-identifier (QI)** adalah atribut yang tidak langsung menunjuk individu tetapi dapat mengidentifikasi bila digabungkan dengan data lain (mis. kebangsaan + waktu + lokasi). Pada data ini, kandidat QI adalah **kebangsaan/paspor**, **bulan**, dan **pintu masuk**. Risiko nyata muncul bila ada sel dengan jumlah sangat kecil (mis. 1-4 orang), karena orang tertentu bisa dikenali.

In [66]:
# ===== 2.5 QUASI-IDENTIFIER: ANALISIS SEL KECIL (k < 5) =====
K_MIN = 5
bulan_cols = BULAN
mask_rinci = ~df_paspor_raw["paspor"].str.contains("TOTAL|A S E A N", regex=True)
paspor_rinci = df_paspor_raw[mask_rinci]
sel_kecil_paspor = ((paspor_rinci[bulan_cols] > 0) & (paspor_rinci[bulan_cols] < K_MIN)).sum().sum()

pintu_rinci = df_pintu_raw[~df_pintu_raw["pintu_masuk"].str.match(r"^[ABC]\. |^Jumlah|^Lainnya$|^Total$")]
sel_kecil_pintu = ((pintu_rinci[bulan_cols] > 0) & (pintu_rinci[bulan_cols] < K_MIN)).sum().sum()

df_qi = pd.DataFrame([
    {"dataset": "BPS-01 Paspor", "kandidat_QI": "paspor (kebangsaan), bulan",
     "sel_rinci_total": int(paspor_rinci[bulan_cols].size), f"sel_dengan_1_sampai_{K_MIN-1}_kunjungan": int(sel_kecil_paspor)},
    {"dataset": "BPS-02 Pengeluaran", "kandidat_QI": "negara tempat tinggal",
     "sel_rinci_total": int(len(df_pengeluaran_raw)), f"sel_dengan_1_sampai_{K_MIN-1}_kunjungan": 0},
    {"dataset": "BPS-03 Pintu Masuk", "kandidat_QI": "pintu masuk, bulan",
     "sel_rinci_total": int(pintu_rinci[bulan_cols].size), f"sel_dengan_1_sampai_{K_MIN-1}_kunjungan": int(sel_kecil_pintu)},
])
display(df_qi)

,dataset,kandidat_QI,sel_rinci_total,sel_dengan_1_sampai_4_kunjungan
0,BPS-01 Paspor,"paspor (kebangsaan), bulan",2880,229
1,BPS-02 Pengeluaran,negara tempat tinggal,37,0
2,BPS-03 Pintu Masuk,"pintu masuk, bulan",348,5


### 2.6 Analisis privasi data
- Seluruh data BPS adalah **statistik agregat** (jumlah kunjungan per kelompok negara/pintu/bulan, rata-rata pengeluaran). Tidak ada baris yang mewakili satu orang, dan tidak ada nama, kontak, atau nomor identitas (lihat 2.4).
- QI (kebangsaan, bulan, pintu masuk) ada, tetapi **berupa kelompok**, bukan individu. Sel kecil (hitungan sangat rendah) pada negara dengan sedikit wisman (lihat 2.5) dicatat sebagai risiko residual rendah. BPS sudah mempublikasikannya sebagai statistik resmi. Analisis hanya dilakukan di level agregat, tanpa mencoba menautkan ke data individu lain.
- Data Open-Meteo hanya berisi **koordinat lokasi publik (bandara/pelabuhan)** dan cuaca, bukan data orang.
- Data Wikipedia adalah informasi publik tentang destinasi; tabel dicek dari pola email/telepon sebelum disimpan.

### 2.7 Data Ethics Checklist 9 butir
Tabel ini diisi di akhir notebook (bagian 6.5) setelah semua bukti (API, robots.txt, scraping, merge) tersedia. Ini adalah **rancangan 9 indikator**; bila dosen memberikan daftar resmi, ganti kolom `indikator` pada sel tersebut.

---
# SOAL 3 · Akuisisi Data Menggunakan API (bobot 25% bersama Soal 4)
## BAGIAN 3: Akuisisi Data Menggunakan API
### 3.1 Pengenalan API Open-Meteo
BPS memiliki API resmi (webapi.bps.go.id) yang memerlukan **App ID/key**. Karena itu notebook ini memakai API publik yang diizinkan soal untuk topik wisata: **Open-Meteo Historical Weather API** (`archive-api.open-meteo.com`). API ini **tidak memerlukan API key** untuk penggunaan gratis non-komersial. Bila suatu saat memakai key, key dibaca dari environment/Secrets Colab, **tidak ditulis di notebook** (lihat sel 3.5).

### 3.2 Pemilihan lokasi pengamatan Indonesia
Lokasi dipilih dari **pintu masuk utama pada dataset BPS-03**, sehingga data cuaca bisa digabung dengan jumlah kunjungan per pintu masuk (kolom penghubung: `pintu_masuk` + `bulan`).

### 3.3 Penentuan koordinat
Koordinat adalah titik bandara/pelabuhan (sumber koordinat: informasi publik bandara). Nama pada kolom `pintu_masuk` **harus sama persis** dengan nama di berkas BPS.

In [67]:
# ===== 3.2-3.3 LOKASI PENGAMATAN DAN KOORDINAT =====
LOKASI = pd.DataFrame([
    # pintu_masuk (nama persis di BPS-03), kota, provinsi, latitude, longitude
    ("Ngurah Rai",            "Denpasar/Badung", "Bali",             -8.7467, 115.1668),
    ("Soekarno-Hatta",        "Tangerang",       "Banten",           -6.1256, 106.6559),
    ("Juanda",                "Surabaya/Sidoarjo","Jawa Timur",      -7.3798, 112.7868),
    ("Kualanamu",             "Deli Serdang",    "Sumatera Utara",    3.6422,  98.8853),
    ("Adi Sucipto",           "Yogyakarta",      "DI Yogyakarta",    -7.7882, 110.4317),
    ("Bandara Int. Lombok",   "Lombok Tengah",   "Nusa Tenggara Barat", -8.7573, 116.2767),
    ("Sam Ratulangi",         "Manado",          "Sulawesi Utara",    1.5493, 124.9259),
    ("Minangkabau",           "Padang Pariaman", "Sumatera Barat",   -0.7869, 100.2806),
    ("Hasanuddin",            "Makassar/Maros",  "Sulawesi Selatan", -5.0617, 119.5540),
    ("Batam",                 "Batam",           "Kepulauan Riau",    1.1301, 104.0529),
], columns=["pintu_masuk", "kota", "provinsi", "latitude", "longitude"])
display(LOKASI)

,pintu_masuk,kota,provinsi,latitude,longitude
0,Ngurah Rai,Denpasar/Badung,Bali,-8.7467,115.1668
1,Soekarno-Hatta,Tangerang,Banten,-6.1256,106.6559
2,Juanda,Surabaya/Sidoarjo,Jawa Timur,-7.3798,112.7868
3,Kualanamu,Deli Serdang,Sumatera Utara,3.6422,98.8853
4,Adi Sucipto,Yogyakarta,DI Yogyakarta,-7.7882,110.4317
5,Bandara Int. Lombok,Lombok Tengah,Nusa Tenggara Barat,-8.7573,116.2767
6,Sam Ratulangi,Manado,Sulawesi Utara,1.5493,124.9259
7,Minangkabau,Padang Pariaman,Sumatera Barat,-0.7869,100.2806
8,Hasanuddin,Makassar/Maros,Sulawesi Selatan,-5.0617,119.5540
9,Batam,Batam,Kepulauan Riau,1.1301,104.0529


### 3.4 Penentuan periode pengambilan
Periode **1 Januari - 31 Desember 2025**, sama dengan periode data BPS. Satu lokasi menghasilkan 365 baris harian, jadi 10 lokasi menghasilkan 3.650 baris (jauh di atas syarat minimal 30 baris).
### 3.5 - 3.10 Request API, timeout, User-Agent, status code, retry, dan jeda
Semua request memakai `http_get()` (sel 0.7b): timeout, User-Agent berisi email kampus, penanganan status 200/4xx/429/5xx, retry dengan exponential backoff, dan jeda 2 detik antar-request.

In [68]:
# ===== 3.5 REQUEST API OPEN-METEO =====
API_URL = "https://archive-api.open-meteo.com/v1/archive"
TGL_MULAI, TGL_AKHIR = "2025-01-01", "2025-12-31"
VARIABEL_HARIAN = "temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max"

# API key OPSIONAL (Open-Meteo gratis tidak perlu key). Bila ada, baca dari environment
# atau Colab Secrets, JANGAN ditulis di notebook dan jangan dicetak.
API_KEY = os.environ.get("OPEN_METEO_API_KEY")
if API_KEY is None:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("OPEN_METEO_API_KEY")
    except Exception:
        API_KEY = None
if API_KEY:
    API_URL = "https://customer-archive-api.open-meteo.com/v1/archive"
print("Mode API:", "berbayar/commercial (key dari env)" if API_KEY else "gratis tanpa key (non-komersial)")

raw_responses = {}      # respons JSON mentah per lokasi (tidak dimodifikasi)
for _, lok in LOKASI.iterrows():
    params = {"latitude": lok.latitude, "longitude": lok.longitude,
              "start_date": TGL_MULAI, "end_date": TGL_AKHIR,
              "daily": VARIABEL_HARIAN, "timezone": "Asia/Jakarta"}
    if API_KEY:
        params["apikey"] = API_KEY
    print(f"Mengambil {lok.pintu_masuk} ...")
    r = http_get(API_URL, params=params)          # timeout + UA + retry + jeda
    raw_responses[lok.pintu_masuk] = r.json()
    print(f"  status={r.status_code}, hari={len(raw_responses[lok.pintu_masuk]['daily']['time'])}")
TANGGAL_API = now_wib()

Mode API: gratis tanpa key (non-komersial)
Mengambil Ngurah Rai ...
  status=200, hari=365
Mengambil Soekarno-Hatta ...
  status=200, hari=365
Mengambil Juanda ...
  status=200, hari=365
Mengambil Kualanamu ...
  status=200, hari=365
Mengambil Adi Sucipto ...
  status=200, hari=365
Mengambil Bandara Int. Lombok ...
  status=200, hari=365
Mengambil Sam Ratulangi ...
  status=200, hari=365
Mengambil Minangkabau ...
  status=200, hari=365
Mengambil Hasanuddin ...
  status=200, hari=365
Mengambil Batam ...
  status=200, hari=365


### 3.11 Penyimpanan respons JSON mentah
Respons asli dari server disimpan **tanpa diubah** ke `data/raw/openmeteo_response.json`. Pembungkus hanya menambahkan kunci nama pintu masuk agar tiap respons dapat dikenali.

In [69]:
# ===== 3.11 SIMPAN JSON MENTAH =====
PATH_API_RAW = RAW / "openmeteo_response.json"
with open(PATH_API_RAW, "w", encoding="utf-8") as f:
    json.dump({"sumber": "Open-Meteo Historical Weather API", "endpoint": API_URL,
               "tanggal_akses": TANGGAL_API, "periode": [TGL_MULAI, TGL_AKHIR],
               "respons_per_pintu_masuk": raw_responses}, f, ensure_ascii=False)
print("Tersimpan:", PATH_API_RAW, "| ukuran:", round(PATH_API_RAW.stat().st_size / 1024, 1), "KB")

catat_provenance("API-01", "Open-Meteo", "Historical Weather API: cuaca harian 2025 di 10 pintu masuk utama",
                 API_URL, "API (HTTP GET, JSON)", PATH_API_RAW,
                 "Data CC BY 4.0; API gratis non-komersial (atribusi Open-Meteo.com)", "JSON",
                 tanggal_akses=TANGGAL_API,
                 catatan="Berisi respons asli per lokasi; tanpa API key")

Tersimpan: UTS_DAM_Pariwisata_Indonesia/data/raw/openmeteo_response.json | ukuran: 159.0 KB


### 3.12 Normalisasi JSON menjadi DataFrame
Struktur respons: `daily` berisi larik paralel (`time`, `temperature_2m_mean`, dst.). Larik ini diubah menjadi satu baris per hari per lokasi.

In [70]:
# ===== 3.12 NORMALISASI JSON -> DATAFRAME =====
potongan = []
for pintu, resp in raw_responses.items():
    d = pd.DataFrame(resp["daily"])                      # larik paralel -> kolom
    d["pintu_masuk"] = pintu
    d["latitude_api"] = resp.get("latitude")
    d["longitude_api"] = resp.get("longitude")
    d["elevation_m"] = resp.get("elevation")
    potongan.append(d)
df_api = pd.concat(potongan, ignore_index=True)
df_api["time"] = pd.to_datetime(df_api["time"])
df_api = df_api.rename(columns={"time": "tanggal"})
kolom_awal = ["pintu_masuk", "tanggal"]
df_api = df_api[kolom_awal + [c for c in df_api.columns if c not in kolom_awal]]

units = next(iter(raw_responses.values())).get("daily_units", {})
print("Satuan dari API:", units)
display(df_api.head())

Satuan dari API: {'time': 'iso8601', 'temperature_2m_mean': '°C', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'precipitation_sum': 'mm', 'wind_speed_10m_max': 'km/h'}


,pintu_masuk,tanggal,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,latitude_api,longitude_api,elevation_m
0,Ngurah Rai,2025-01-01,26.6,28.1,25.1,29.0,10.6,-8.822495,115.159615,3.0
1,Ngurah Rai,2025-01-02,25.7,26.7,25.0,33.5,9.7,-8.822495,115.159615,3.0
2,Ngurah Rai,2025-01-03,25.9,27.7,24.2,19.0,18.0,-8.822495,115.159615,3.0
3,Ngurah Rai,2025-01-04,25.6,27.2,24.1,18.8,16.5,-8.822495,115.159615,3.0
4,Ngurah Rai,2025-01-05,25.7,27.9,23.8,38.1,12.8,-8.822495,115.159615,3.0


### 3.13 Pemeriksaan minimal 30 baris

In [71]:
print("Jumlah baris hasil API :", len(df_api))
assert len(df_api) >= 30, "Syarat soal: tabel API minimal 30 baris"
print("Syarat minimal 30 baris terpenuhi [OK]")
print("\nBaris per lokasi:")
display(df_api.groupby("pintu_masuk").size().rename("jumlah_hari").to_frame())
print("Nilai kosong per kolom:")
display(df_api.isna().sum().to_frame("jumlah_kosong"))

Jumlah baris hasil API : 3650
Syarat minimal 30 baris terpenuhi [OK]

Baris per lokasi:


,jumlah_hari
pintu_masuk,
Adi Sucipto,365
Bandara Int. Lombok,365
Batam,365
Hasanuddin,365
Juanda,365
Kualanamu,365
Minangkabau,365
Ngurah Rai,365
Sam Ratulangi,365


Nilai kosong per kolom:


,jumlah_kosong
pintu_masuk,0
tanggal,0
temperature_2m_mean,0
temperature_2m_max,0
temperature_2m_min,0
precipitation_sum,0
wind_speed_10m_max,0
latitude_api,0
longitude_api,0
elevation_m,0


### 3.14 Dokumentasi hasil API

In [72]:
df_api.to_csv(PROC / "openmeteo_clean.csv", index=False)
print("Tersimpan: data/processed/openmeteo_clean.csv")
display(df_api.describe().T)
print("\nLog request API (bukti status code, retry, dan durasi):")
df_log = pd.DataFrame(REQUEST_LOG)
display(df_log[df_log["url"].str.contains("open-meteo")].tail(12))

Tersimpan: data/processed/openmeteo_clean.csv


,count,mean,min,25%,50%,75%,max,std
tanggal,3650,2025-07-02 00:00:00,2025-01-01 00:00:00,2025-04-02 00:00:00,2025-07-02 00:00:00,2025-10-01 00:00:00,2025-12-31 00:00:00,NaN
temperature_2m_mean,3650.0,26.489479,22.9,25.6,26.4,27.3,30.7,1.238352
temperature_2m_max,3650.0,30.228,23.8,29.2,30.2,31.2,36.6,1.648509
temperature_2m_min,3650.0,23.675616,17.5,22.8,23.5,24.6,28.5,1.321951
precipitation_sum,3650.0,8.022082,0.0,0.9,4.0,11.6,112.0,10.494524
wind_speed_10m_max,3650.0,13.166521,4.2,10.1,12.3,15.4,34.7,4.31851
latitude_api,3650.0,-3.831283,-8.822495,-7.768014,-5.623901,1.15993,3.620387,4.509175
longitude_api,3650.0,110.912769,98.905106,104.05363,111.636787,116.30172,124.94453,8.034219
elevation_m,3650.0,34.0,2.0,3.0,10.0,82.0,118.0,42.441648



Log request API (bukti status code, retry, dan durasi):


,waktu,url,percobaan,status,galat,durasi_detik
1,2026-11-19 15:34:06 WIB,https://open-meteo.com/en/license,1,200,,0.07
2,2026-11-19 15:34:09 WIB,https://open-meteo.com/en/terms,1,200,,0.06
4,2026-11-19 15:34:14 WIB,https://archive-api.open-meteo.com/v1/archive,1,200,,0.63
5,2026-11-19 15:34:17 WIB,https://archive-api.open-meteo.com/v1/archive,1,200,,0.25
6,2026-11-19 15:34:19 WIB,https://archive-api.open-meteo.com/v1/archive,1,200,,0.16
7,2026-11-19 15:34:21 WIB,https://archive-api.open-meteo.com/v1/archive,1,200,,0.18
8,2026-11-19 15:34:23 WIB,https://archive-api.open-meteo.com/v1/archive,1,200,,0.16
9,2026-11-19 15:34:25 WIB,https://archive-api.open-meteo.com/v1/archive,1,200,,0.16
10,2026-11-19 15:34:27 WIB,https://archive-api.open-meteo.com/v1/archive,1,200,,0.24
11,2026-11-19 15:34:30 WIB,https://archive-api.open-meteo.com/v1/archive,1,200,,0.17


---
# SOAL 4 · Web Scraping Wikipedia (bobot 20% bersama Soal 2)
## BAGIAN 4: Web Scraping Wikipedia
### 4.1 Penentuan halaman Wikipedia
Halaman statis: **"Pariwisata di Indonesia"** (Wikipedia bahasa Indonesia). Wikipedia diizinkan oleh soal (bukan situs yang dilarang) dan berlisensi CC BY-SA 4.0.
### 4.2 - 4.4 Pemeriksaan robots.txt, analisis aturan, dan bukti kepatuhan

In [73]:

# ===== 4.2-4.4 PEMERIKSAAN ROBOTS.TXT DIREKTORI PARIWISATA =====

# URL website yang menjadi sumber scraping
URL_DIREKTORI = "https://www.direktoripariwisata.id/"
URL_ROBOTS_DIREKTORI = URL_DIREKTORI.rstrip("/") + "/robots.txt"

print("Memeriksa robots.txt Direktori Pariwisata Indonesia...")

# Mengambil robots.txt menggunakan fungsi http_get()
r_robots_direktori = http_get(URL_ROBOTS_DIREKTORI)

print("Status HTTP:", r_robots_direktori.status_code)

if r_robots_direktori.status_code != 200:
    raise RuntimeError(
        "robots.txt tidak berhasil diperiksa. "
        "Jangan lanjutkan scraping sebelum ketentuan akses dapat diverifikasi."
    )

TEKS_ROBOTS_DIREKTORI = r_robots_direktori.text

# Menyimpan bukti robots.txt
PATH_ROBOTS_DIREKTORI = DOC / "bukti_robots_direktori.txt"

PATH_ROBOTS_DIREKTORI.write_text(
    f"URL: {URL_ROBOTS_DIREKTORI}\n"
    f"Tanggal akses: {now_wib()}\n"
    f"Status HTTP: {r_robots_direktori.status_code}\n\n"
    f"{TEKS_ROBOTS_DIREKTORI}",
    encoding="utf-8"
)

# Memeriksa aturan robots.txt
rp_direktori = RobotFileParser()
rp_direktori.parse(TEKS_ROBOTS_DIREKTORI.splitlines())

boleh_direktori = rp_direktori.can_fetch(
    USER_AGENT, URL_DIREKTORI
)

crawl_delay_direktori = (
    rp_direktori.crawl_delay(USER_AGENT)
    or rp_direktori.crawl_delay("*")
)

print("User-Agent:", USER_AGENT)
print("URL target:", URL_DIREKTORI)
print("Izin scraping:", boleh_direktori)
print("Crawl delay:", crawl_delay_direktori)

# Menyesuaikan jeda bila robots.txt menetapkan crawl-delay
if crawl_delay_direktori:
    JEDA_DETIK = max(
        JEDA_DETIK,
        float(crawl_delay_direktori)
    )

assert boleh_direktori, (
    "robots.txt tidak mengizinkan akses ke halaman target. "
    "Scraping dihentikan."
)

print("Pemeriksaan robots.txt selesai.")


Memeriksa robots.txt Direktori Pariwisata Indonesia...
Status HTTP: 200
User-Agent: UTS-DAM-INSTIKI/1.0 (tugas kuliah; PUTU INTAN CAHYANTI PUTRI; NIM 2501010046; kontak: mahasiswa@instiki.ac.id) python-requests/2.32.4
URL target: https://www.direktoripariwisata.id/
Izin scraping: True
Crawl delay: None
Pemeriksaan robots.txt selesai.


### 4.5 Pengambilan halaman HTML dan penyimpanan HTML mentah (4.9)

In [74]:

# ===== 4.5 + 4.9 AKUISISI HTML DIREKTORI PARIWISATA =====

# Mengambil halaman utama website
r_direktori = http_get(URL_DIREKTORI)

assert r_direktori.status_code == 200, (
    "Halaman Direktori Pariwisata gagal diakses."
)

TANGGAL_DIREKTORI = now_wib()

# Menyimpan HTML mentah tanpa proses pembersihan
PATH_DIREKTORI_RAW = RAW / "direktori_pariwisata.html"

PATH_DIREKTORI_RAW.write_bytes(r_direktori.content)

print("Status HTTP:", r_direktori.status_code)
print("Ukuran HTML:", round(
    PATH_DIREKTORI_RAW.stat().st_size / 1024, 2
), "KB")

# Mencatat provenance sumber web scraping
catat_provenance(
    "WEB-01",
    "Direktori Pariwisata Indonesia",
    "Halaman utama dan tautan destinasi wisata",
    URL_DIREKTORI,
    "Web scraping menggunakan BeautifulSoup",
    PATH_DIREKTORI_RAW,
    "Ketentuan penggunaan website harus diperiksa; cantumkan atribusi sumber",
    "HTML",
    tanggal_akses=TANGGAL_DIREKTORI,
    catatan="HTML mentah disimpan sebelum proses ekstraksi."
)

# Membaca HTML untuk proses ekstraksi
soup_direktori = BeautifulSoup(
    r_direktori.text,
    "html.parser"
)

print(
    "Judul website:",
    soup_direktori.title.get_text(strip=True)
    if soup_direktori.title else "Tidak ditemukan"
)


Status HTTP: 200
Ukuran HTML: 35.76 KB
Judul website: Direktori Pariwisata


### 4.6 - 4.8 Ekstraksi tabel/listing dengan `pd.read_html` dan BeautifulSoup
Halaman dibaca dari berkas HTML mentah (bukan mengambil ulang dari web). `pd.read_html` dipakai untuk tabel; bila tidak ada tabel yang layak, BeautifulSoup dipakai untuk mengambil listing (daftar bertingkat di bawah judul bagian).

In [75]:

# ===== 4.6 EKSTRAKSI TAUTAN INTERNAL WEBSITE =====

from urllib.parse import urljoin, urlparse

domain_direktori = urlparse(URL_DIREKTORI).netloc

daftar_link = []
link_tercatat = set()

# Mengambil semua elemen anchor yang memiliki href
for a in soup_direktori.find_all("a", href=True):

    nama_link = a.get_text(" ", strip=True)
    url_link = urljoin(URL_DIREKTORI, a["href"])

    parsed_url = urlparse(url_link)

    # Hanya mengambil tautan dari domain yang sama
    if parsed_url.netloc != domain_direktori:
        continue

    # Mengabaikan tautan kosong
    if not nama_link:
        continue

    # Menghindari tautan duplikat
    if url_link in link_tercatat:
        continue

    link_tercatat.add(url_link)

    daftar_link.append({
        "nama_link": nama_link,
        "url": url_link
    })

df_link_direktori = pd.DataFrame(daftar_link)

print("Jumlah tautan internal:", len(df_link_direktori))

# Menampilkan hasil ekstraksi
display(df_link_direktori.head(30))

# Menyimpan seluruh tautan internal sebagai data hasil ekstraksi awal
df_link_direktori.to_csv(
    PROC / "tautan_direktori_pariwisata.csv",
    index=False,
    encoding="utf-8-sig"
)


Jumlah tautan internal: 29


,nama_link,url
0,Depan,https://www.direktoripariwisata.id/
1,Wisata Alam,https://www.direktoripariwisata.id/browse/cat-1
2,Wisata Buatan,https://www.direktoripariwisata.id/browse/cat-2
3,Wisata Budaya,https://www.direktoripariwisata.id/browse/cat-3
4,Taman Nasional,https://www.direktoripariwisata.id/browse/cat-4
5,Bromo,https://www.direktoripariwisata.id/unit/1
6,Candi Borobudur,https://www.direktoripariwisata.id/unit/10
7,Danau Toba,https://www.direktoripariwisata.id/unit/3
8,Kepulauan Seribu,https://www.direktoripariwisata.id/unit/8
9,Labuan Bajo,https://www.direktoripariwisata.id/unit/2


In [76]:

# ===== 4.7 EKSTRAKSI INFORMASI HALAMAN DESTINASI =====

# Kata kunci untuk menyaring tautan yang berpotensi
# berkaitan dengan destinasi pariwisata
KATA_KUNCI_WISATA = [
    "destinasi",
    "wisata",
    "pariwisata",
    "alam",
    "budaya",
    "pantai",
    "gunung",
    "danau",
    "taman"
]

if not df_link_direktori.empty:

    pola_wisata = "|".join(KATA_KUNCI_WISATA)

    kandidat_link = df_link_direktori[
        df_link_direktori["nama_link"].str.contains(
            pola_wisata,
            case=False,
            na=False
        )
        |
        df_link_direktori["url"].str.contains(
            pola_wisata,
            case=False,
            na=False
        )
    ].copy()

else:
    kandidat_link = pd.DataFrame(
        columns=["nama_link", "url"]
    )

print("Jumlah kandidat halaman wisata:", len(kandidat_link))

display(kandidat_link.head(30))

# Batasi jumlah halaman yang diambil
BATAS_HALAMAN = 30

hasil_direktori = []

# Folder untuk menyimpan HTML mentah setiap halaman
FOLDER_DETAIL_RAW = RAW / "direktori_detail"
FOLDER_DETAIL_RAW.mkdir(parents=True, exist_ok=True)

for nomor, (_, baris) in enumerate(
    kandidat_link.head(BATAS_HALAMAN).iterrows(),
    start=1
):

    url = baris["url"]

    try:
        # Menggunakan http_get agar timeout, User-Agent,
        # retry, dan jeda tetap diterapkan
        response_detail = http_get(url)

        if response_detail.status_code != 200:
            print("Gagal mengambil:", url)
            continue

        # Simpan HTML mentah halaman detail
        nama_file = f"halaman_{nomor:03d}.html"

        path_html = FOLDER_DETAIL_RAW / nama_file

        path_html.write_bytes(response_detail.content)

        # Membaca HTML untuk ekstraksi
        detail_soup = BeautifulSoup(
            response_detail.text,
            "html.parser"
        )

        # Mengambil judul halaman
        h1 = detail_soup.find("h1")

        if h1:
            judul = h1.get_text(" ", strip=True)
        elif detail_soup.title:
            judul = detail_soup.title.get_text(" ", strip=True)
        else:
            judul = baris["nama_link"]

        # Mengambil meta description
        meta = detail_soup.find(
            "meta",
            attrs={"name": "description"}
        )

        deskripsi = (
            meta.get("content", "").strip()
            if meta else ""
        )

        # Mengambil teks dari bagian utama jika meta description kosong
        if not deskripsi:

            konten = (
                detail_soup.find("main")
                or detail_soup.find("article")
            )

            if konten:
                deskripsi = konten.get_text(
                    " ", strip=True
                )[:500]

        hasil_direktori.append({
            "nama_destinasi": judul,
            "nama_link": baris["nama_link"],
            "deskripsi": deskripsi,
            "url_sumber": url,
            "file_html_raw": nama_file,
            "tanggal_akses": now_wib()
        })

        print(f"{nomor}. Berhasil mengambil: {judul}")

    except Exception as e:
        print("Kesalahan pada URL:", url)
        print(type(e).__name__, e)

# Membentuk DataFrame hasil scraping
df_direktori = pd.DataFrame(hasil_direktori)

print("\nJumlah halaman berhasil diambil:", len(df_direktori))

display(df_direktori.head(20))


Jumlah kandidat halaman wisata: 29


,nama_link,url
0,Depan,https://www.direktoripariwisata.id/
1,Wisata Alam,https://www.direktoripariwisata.id/browse/cat-1
2,Wisata Buatan,https://www.direktoripariwisata.id/browse/cat-2
3,Wisata Budaya,https://www.direktoripariwisata.id/browse/cat-3
4,Taman Nasional,https://www.direktoripariwisata.id/browse/cat-4
5,Bromo,https://www.direktoripariwisata.id/unit/1
6,Candi Borobudur,https://www.direktoripariwisata.id/unit/10
7,Danau Toba,https://www.direktoripariwisata.id/unit/3
8,Kepulauan Seribu,https://www.direktoripariwisata.id/unit/8
9,Labuan Bajo,https://www.direktoripariwisata.id/unit/2


1. Berhasil mengambil: Jakarta
2. Berhasil mengambil: Direktori
3. Berhasil mengambil: Direktori
4. Berhasil mengambil: Direktori
5. Berhasil mengambil: Direktori
6. Berhasil mengambil: Bromo
7. Berhasil mengambil: Candi Borobudur
8. Berhasil mengambil: Danau Toba
9. Berhasil mengambil: Kepulauan Seribu
10. Berhasil mengambil: Labuan Bajo
11. Berhasil mengambil: Likupang
12. Berhasil mengambil: Mandalika
13. Berhasil mengambil: Morotai
14. Berhasil mengambil: Tanjung Kelayang
15. Berhasil mengambil: Wakatobi
16. Berhasil mengambil: Bandung
17. Berhasil mengambil: Banyuwangi
18. Berhasil mengambil: Jakarta
19. Berhasil mengambil: Joglosemar
20. Berhasil mengambil: Kepulauan Riau
21. Berhasil mengambil: Lombok
22. Berhasil mengambil: Makassar
23. Berhasil mengambil: Medan
24. Berhasil mengambil: Pulau Bali
25. Berhasil mengambil: Wakatobi-Bunaken-Rajaampat
26. Berhasil mengambil: Kementerian Pariwisata Republik Indonesia
27. Berhasil mengambil: Jakarta
28. Berhasil mengambil: Jakarta
Kes

,nama_destinasi,nama_link,deskripsi,url_sumber,file_html_raw,tanggal_akses
0,Jakarta,Depan,,https://www.direktoripariwisata.id/,halaman_001.html,2026-11-19 15:34:43 WIB
1,Direktori,Wisata Alam,,https://www.direktoripariwisata.id/browse/cat-1,halaman_002.html,2026-11-19 15:34:46 WIB
2,Direktori,Wisata Buatan,,https://www.direktoripariwisata.id/browse/cat-2,halaman_003.html,2026-11-19 15:34:48 WIB
3,Direktori,Wisata Budaya,,https://www.direktoripariwisata.id/browse/cat-3,halaman_004.html,2026-11-19 15:34:50 WIB
4,Direktori,Taman Nasional,,https://www.direktoripariwisata.id/browse/cat-4,halaman_005.html,2026-11-19 15:34:53 WIB
5,Bromo,Bromo,,https://www.direktoripariwisata.id/unit/1,halaman_006.html,2026-11-19 15:34:55 WIB
6,Candi Borobudur,Candi Borobudur,,https://www.direktoripariwisata.id/unit/10,halaman_007.html,2026-11-19 15:34:57 WIB
7,Danau Toba,Danau Toba,,https://www.direktoripariwisata.id/unit/3,halaman_008.html,2026-11-19 15:34:59 WIB
8,Kepulauan Seribu,Kepulauan Seribu,,https://www.direktoripariwisata.id/unit/8,halaman_009.html,2026-11-19 15:35:02 WIB
9,Labuan Bajo,Labuan Bajo,,https://www.direktoripariwisata.id/unit/2,halaman_010.html,2026-11-19 15:35:04 WIB


### 4.10 Dokumentasi hasil scraping

In [77]:

# ===== 4.8 PEMBERSIHAN DAN PENYIMPANAN DATA SCRAPING =====

if not df_direktori.empty:

    # Menghapus baris duplikat berdasarkan URL
    df_direktori = df_direktori.drop_duplicates(
        subset=["url_sumber"]
    )

    # Menghapus data yang tidak memiliki nama destinasi
    df_direktori = df_direktori[
        df_direktori["nama_destinasi"].notna()
        &
        (df_direktori["nama_destinasi"].str.strip() != "")
    ]

    df_direktori = df_direktori.reset_index(drop=True)

# Simpan hasil akhir ke CSV
PATH_DIREKTORI_CLEAN = PROC / "direktori_pariwisata_clean.csv"

df_direktori.to_csv(
    PATH_DIREKTORI_CLEAN,
    index=False,
    encoding="utf-8-sig"
)

print("========== HASIL WEB SCRAPING ==========")
print("Jumlah baris:", len(df_direktori))
print("Jumlah kolom:", len(df_direktori.columns))
print("Kolom:", list(df_direktori.columns))

display(df_direktori.head(20))

print("\nFile tersimpan:", PATH_DIREKTORI_CLEAN)


========== HASIL WEB SCRAPING ==========
Jumlah baris: 28
Jumlah kolom: 6
Kolom: ['nama_destinasi', 'nama_link', 'deskripsi', 'url_sumber', 'file_html_raw', 'tanggal_akses']


,nama_destinasi,nama_link,deskripsi,url_sumber,file_html_raw,tanggal_akses
0,Jakarta,Depan,,https://www.direktoripariwisata.id/,halaman_001.html,2026-11-19 15:34:43 WIB
1,Direktori,Wisata Alam,,https://www.direktoripariwisata.id/browse/cat-1,halaman_002.html,2026-11-19 15:34:46 WIB
2,Direktori,Wisata Buatan,,https://www.direktoripariwisata.id/browse/cat-2,halaman_003.html,2026-11-19 15:34:48 WIB
3,Direktori,Wisata Budaya,,https://www.direktoripariwisata.id/browse/cat-3,halaman_004.html,2026-11-19 15:34:50 WIB
4,Direktori,Taman Nasional,,https://www.direktoripariwisata.id/browse/cat-4,halaman_005.html,2026-11-19 15:34:53 WIB
5,Bromo,Bromo,,https://www.direktoripariwisata.id/unit/1,halaman_006.html,2026-11-19 15:34:55 WIB
6,Candi Borobudur,Candi Borobudur,,https://www.direktoripariwisata.id/unit/10,halaman_007.html,2026-11-19 15:34:57 WIB
7,Danau Toba,Danau Toba,,https://www.direktoripariwisata.id/unit/3,halaman_008.html,2026-11-19 15:34:59 WIB
8,Kepulauan Seribu,Kepulauan Seribu,,https://www.direktoripariwisata.id/unit/8,halaman_009.html,2026-11-19 15:35:02 WIB
9,Labuan Bajo,Labuan Bajo,,https://www.direktoripariwisata.id/unit/2,halaman_010.html,2026-11-19 15:35:04 WIB



File tersimpan: UTS_DAM_Pariwisata_Indonesia/data/processed/direktori_pariwisata_clean.csv


In [78]:

# ===== 4.10 DOKUMENTASI HASIL SCRAPING =====

print("Sumber:", URL_DIREKTORI)
print("Tanggal akses:", TANGGAL_DIREKTORI)
print("Metode: requests + BeautifulSoup")
print("Jumlah tautan internal:", len(df_link_direktori))
print("Jumlah kandidat halaman:", len(kandidat_link))
print("Jumlah halaman berhasil diambil:", len(df_direktori))

# Pemeriksaan nilai kosong
print("\nNilai kosong:")
display(df_direktori.isna().sum().to_frame("jumlah_kosong"))

# Pemeriksaan duplikasi URL
print(
    "\nDuplikasi URL:",
    df_direktori["url_sumber"].duplicated().sum()
    if not df_direktori.empty else 0
)

# Menampilkan log request website
df_log_direktori = pd.DataFrame(REQUEST_LOG)

display(
    df_log_direktori[
        df_log_direktori["url"].str.contains(
            "direktoripariwisata.id",
            na=False
        )
    ]
)

print("\nHTML mentah:", PATH_DIREKTORI_RAW)
print("CSV hasil scraping:", PATH_DIREKTORI_CLEAN)
print("Bukti robots.txt:", PATH_ROBOTS_DIREKTORI)


Sumber: https://www.direktoripariwisata.id/
Tanggal akses: 2026-11-19 15:34:41 WIB
Metode: requests + BeautifulSoup
Jumlah tautan internal: 29
Jumlah kandidat halaman: 29
Jumlah halaman berhasil diambil: 28

Nilai kosong:


,jumlah_kosong
nama_destinasi,0
nama_link,0
deskripsi,0
url_sumber,0
file_html_raw,0
tanggal_akses,0



Duplikasi URL: 0


,waktu,url,percobaan,status,galat,durasi_detik
14,2026-11-19 15:34:36 WIB,https://www.direktoripariwisata.id/robots.txt,1,200,,0.09
15,2026-11-19 15:34:39 WIB,https://www.direktoripariwisata.id/,1,200,,0.75
16,2026-11-19 15:34:41 WIB,https://www.direktoripariwisata.id/,1,200,,0.26
17,2026-11-19 15:34:44 WIB,https://www.direktoripariwisata.id/browse/cat-1,1,200,,0.28
18,2026-11-19 15:34:46 WIB,https://www.direktoripariwisata.id/browse/cat-2,1,200,,0.26
19,2026-11-19 15:34:48 WIB,https://www.direktoripariwisata.id/browse/cat-3,1,200,,0.26
20,2026-11-19 15:34:51 WIB,https://www.direktoripariwisata.id/browse/cat-4,1,200,,0.25
21,2026-11-19 15:34:53 WIB,https://www.direktoripariwisata.id/unit/1,1,200,,0.28
22,2026-11-19 15:34:55 WIB,https://www.direktoripariwisata.id/unit/10,1,200,,0.26
23,2026-11-19 15:34:57 WIB,https://www.direktoripariwisata.id/unit/3,1,200,,0.29



HTML mentah: UTS_DAM_Pariwisata_Indonesia/data/raw/direktori_pariwisata.html
CSV hasil scraping: UTS_DAM_Pariwisata_Indonesia/data/processed/direktori_pariwisata_clean.csv
Bukti robots.txt: UTS_DAM_Pariwisata_Indonesia/documentation/bukti_robots_direktori.txt


---
# SOAL 5 · Integrasi dan Penggabungan Data (bobot 25%)
## BAGIAN 5: Integrasi dan Penggabungan Data
### 5.1 - 5.2 Persiapan dataset dan pemeriksaan struktur
Dua penggabungan dilakukan, keduanya berasal dari berkas dan API:
- **Merge A (utama):** BPS-03 kunjungan per pintu masuk (berkas) + Open-Meteo cuaca bulanan (API), kunci `pintu_masuk` + `bulan_no` (relasi 1:1).
- **Merge B (pendukung):** BPS-01 kunjungan menurut paspor + BPS-02 pengeluaran per kunjungan (keduanya berkas BPS), kunci nama negara (1:1).

Wikipedia tetap disimpan sebagai sumber ketiga yang berdiri sendiri: halamannya tidak punya kolom yang cocok secara valid dengan kunci di atas, sehingga tidak digabung secara dipaksakan.

Langkah pertama adalah merapikan (tidy) data BPS menjadi format yang siap digabung. Data mentah di `data/raw` tidak disentuh.

In [79]:
# ===== 5.1 PERSIAPAN: RAPIKAN BPS-03 (PINTU MASUK) KE FORMAT LONG =====
KATEGORI = {"A": "Udara", "B": "Laut", "C": "Darat"}
baris_pintu, kategori_aktif = [], "-"
for _, r in df_pintu_raw.iterrows():
    nama = r["pintu_masuk"]
    m = re.match(r"^([ABC])\. (Pintu \w+)", nama)
    if m:
        kategori_aktif, level = KATEGORI[m.group(1)], "subtotal"      # baris subtotal kategori
    elif nama in ("Jumlah (A+B+C)", "Lainnya", "Total"):
        kategori_aktif, level = "Total", "total"
    else:
        level = "pintu"                                                # baris pintu rinci
    d = {"pintu_masuk": nama, "kategori_pintu": kategori_aktif, "level": level, "tahunan_bps": r["Tahunan"]}
    d.update({b: r[b] for b in BULAN})
    baris_pintu.append(d)
df_pintu_wide = pd.DataFrame(baris_pintu)

# Cek kualitas: jumlah 12 bulan harus sama dengan kolom Tahunan dari BPS
selisih = (df_pintu_wide[BULAN].sum(axis=1) - df_pintu_wide["tahunan_bps"]).abs()
print("Baris dengan jumlah 12 bulan != Tahunan:", int((selisih != 0).sum()))

# wide -> long (satu baris per pintu per bulan)
df_pintu_long = df_pintu_wide.melt(
    id_vars=["pintu_masuk", "kategori_pintu", "level", "tahunan_bps"],
    value_vars=BULAN, var_name="bulan_nama", value_name="kunjungan")
df_pintu_long["bulan_no"] = df_pintu_long["bulan_nama"].map({b: i + 1 for i, b in enumerate(BULAN)})
df_pintu_long["tahun"] = 2025
df_pintu_long = df_pintu_long.sort_values(["pintu_masuk", "bulan_no"]).reset_index(drop=True)
print("BPS-03 format long:", df_pintu_long.shape)
display(df_pintu_long.head(6))

Baris dengan jumlah 12 bulan != Tahunan: 0
BPS-03 format long: (420, 8)


,pintu_masuk,kategori_pintu,level,tahunan_bps,bulan_nama,kunjungan,bulan_no,tahun
0,A. Pintu Udara,Udara,subtotal,10819291,Januari,801614,1,2025
1,A. Pintu Udara,Udara,subtotal,10819291,Februari,718609,2,2025
2,A. Pintu Udara,Udara,subtotal,10819291,Maret,669479,3,2025
3,A. Pintu Udara,Udara,subtotal,10819291,April,854756,4,2025
4,A. Pintu Udara,Udara,subtotal,10819291,Mei,941092,5,2025
5,A. Pintu Udara,Udara,subtotal,10819291,Juni,978929,6,2025


In [80]:
# ===== 5.1 PERSIAPAN: RAPIKAN BPS-01 (PASPOR) DAN BPS-02 (PENGELUARAN) =====
norm = lambda s: re.sub(r"\s+", "", str(s)).lower()      # "I t a l y" -> "italy"

# Tentukan wilayah tiap baris: baris sebelum subtotal 'TOTAL ...' / 'A S E A N' masuk wilayah tsb
WILAYAH_TOTAL = {"asean": "ASEAN", "totalasia(excl.asean)": "Asia (di luar ASEAN)",
                 "totalmiddleeast": "Timur Tengah", "totaleurope": "Eropa",
                 "totalamerica": "Amerika", "totaloceania": "Oseania", "totalafrica": "Afrika"}
wilayah, tertunda = [], []
for nama in df_paspor_raw["paspor"]:
    tertunda.append(len(wilayah))
    wilayah.append(None)
    k = norm(nama)
    if k in WILAYAH_TOTAL:
        for i in tertunda:
            wilayah[i] = WILAYAH_TOTAL[k]
        tertunda = []
    elif k == "grandtotal":
        wilayah[-1] = "Total"
        tertunda = []

df_paspor = df_paspor_raw.copy()
df_paspor["wilayah_asal"] = wilayah
df_paspor["is_agregat"] = df_paspor["paspor"].map(lambda s: norm(s) in WILAYAH_TOTAL or norm(s).startswith("total") or norm(s) == "grandtotal")
df_paspor["negara_key"] = df_paspor["paspor"].map(norm)
print("BPS-01:", df_paspor.shape, "| baris agregat:", int(df_paspor["is_agregat"].sum()))
print("Duplikat nama paspor:", int(df_paspor["paspor"].duplicated().sum()))

# Cek kualitas: jumlah baris rinci vs GRAND TOTAL
rinci = df_paspor[~df_paspor["is_agregat"]]
gt = df_paspor.loc[df_paspor["negara_key"] == "grandtotal", "Tahunan"].iloc[0]
print(f"Jumlah 'Tahunan' baris rinci = {rinci['Tahunan'].sum():,} | GRAND TOTAL BPS = {gt:,}")
print("Catatan: selisih ini wajar. Dalam format BPS, baris kelompok seperti 'Other Asia', 'Other Middle East', "
      "'Central America', 'South America', dan 'Other America' tampaknya merupakan subtotal dari baris di bawahnya, "
      "sehingga menjumlahkan semua baris non-TOTAL menghitung sebagian wisman dua kali. "
      "Karena itu JANGAN menjumlahkan seluruh baris untuk mendapat total nasional; pakai baris GRAND TOTAL. "
      "Untuk Merge B, baris 'Central America' dan 'South America' justru dipakai karena BPS-02 juga melaporkan "
      "'Amerika Tengah' dan 'Amerika Selatan' sebagai kelompok.")
display(df_paspor.head(6))

BPS-01: (248, 17) | baris agregat: 8
Duplikat nama paspor: 0
Jumlah 'Tahunan' baris rinci = 15,533,325 | GRAND TOTAL BPS = 15,386,646
Catatan: selisih ini wajar. Dalam format BPS, baris kelompok seperti 'Other Asia', 'Other Middle East', 'Central America', 'South America', dan 'Other America' tampaknya merupakan subtotal dari baris di bawahnya, sehingga menjumlahkan semua baris non-TOTAL menghitung sebagian wisman dua kali. Karena itu JANGAN menjumlahkan seluruh baris untuk mendapat total nasional; pakai baris GRAND TOTAL. Untuk Merge B, baris 'Central America' dan 'South America' justru dipakai karena BPS-02 juga melaporkan 'Amerika Tengah' dan 'Amerika Selatan' sebagai kelompok.


,paspor,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,wilayah_asal,is_agregat,negara_key
0,Malaysia,221474,214605,159895,170018,238497,236417,212113,229726,272421,234384,207037,243162,2639749,ASEAN,False,malaysia
1,Singapore,110484,89757,100883,97506,126391,183748,114224,129864,119237,123578,126254,204512,1526438,ASEAN,False,singapore
2,Timor Leste,75735,61770,63471,69721,80809,82296,90840,89516,86861,90077,98987,119098,1009181,ASEAN,False,timorleste
3,Indonesia,46451,42534,35310,45118,51180,69252,55391,62962,54550,58476,50980,59812,632016,ASEAN,False,indonesia
4,Philippines,16370,15505,15457,18652,23264,24580,22132,22278,20437,22948,22054,23051,246728,ASEAN,False,philippines
5,Thailand,6043,7262,6536,10073,11683,9490,11397,11953,10099,11124,8423,8054,112137,ASEAN,False,thailand


In [81]:
# ===== 5.3 PENYERAGAMAN NAMA NEGARA (paspor berbahasa Inggris vs pengeluaran berbahasa Indonesia) =====
# Pemetaan eksplisit nama paspor -> nama pada BPS-02 (kunci dinormalkan: huruf kecil tanpa spasi)
PETA_NEGARA = {
    "malaysia": "Malaysia", "singapore": "Singapura", "philippines": "Filipina", "thailand": "Thailand",
    "bruneidarussalam": "Brunei Darussalam", "india": "India", "southkorea": "Korea Selatan",
    "japan": "Jepang", "pakistan": "Pakistan", "bangladesh": "Bangladesh", "srilanka": "Srilanka",
    "republikrakyattiongkok": "Republik Rakyat Tiongkok", "saudiarabia": "Arab Saudi",
    "austria": "Austria", "belgium": "Belgia", "denmark": "Denmark", "france": "Perancis",
    "germany": "Jerman", "italy": "Italia", "netherlands": "Belanda", "spain": "Spanyol",
    "portugal": "Portugal", "sweden": "Swedia", "switzerland": "Swiss", "unitedkingdom": "Inggris",
    "finland": "Finlandia", "norway": "Norwegia", "rusia": "Rusia",
    "unitedstatesofamerica": "Amerika Serikat", "canada": "Kanada", "centralamerica": "Amerika Tengah",
    "southamerica": "Amerika Selatan", "australia": "Australia", "newzealand": "Selandia Baru",
    "egypt": "Mesir",
}
df_paspor["negara_std"] = df_paspor["negara_key"].map(PETA_NEGARA)

df_pengeluaran = df_pengeluaran_raw.copy()
df_pengeluaran["negara_std"] = df_pengeluaran["negara"].str.strip()
df_pengeluaran["is_agregat"] = df_pengeluaran["negara_std"].isin(["Rata- Rata", "Negara lainnya"])

# Pastikan SEMUA negara rinci pada BPS-02 punya pasangan di BPS-01
negara_rinci_02 = set(df_pengeluaran.loc[~df_pengeluaran["is_agregat"], "negara_std"])
negara_terpetakan = set(df_paspor["negara_std"].dropna())
tak_cocok = negara_rinci_02 - negara_terpetakan
print("Negara BPS-02 (rinci):", len(negara_rinci_02), "| terpetakan ke BPS-01:", len(negara_rinci_02 & negara_terpetakan))
print("Negara BPS-02 tanpa pasangan:", tak_cocok if tak_cocok else "tidak ada")
assert not tak_cocok, "Perbaiki PETA_NEGARA sebelum merge"

Negara BPS-02 (rinci): 35 | terpetakan ke BPS-01: 35
Negara BPS-02 tanpa pasangan: tidak ada


### 5.4 Penentuan kolom penghubung
| Merge | Tabel kiri | Tabel kanan | Kolom penghubung | Relasi |
|---|---|---|---|---|
| A | BPS-03 (pintu masuk, long) | Open-Meteo (agregat bulanan) | `pintu_masuk` + `bulan_no` | 1:1 |
| B | BPS-01 (paspor) | BPS-02 (pengeluaran) | `negara_std` (hasil penyeragaman nama) | 1:1 |

Cuaca harian dari API diagregasi ke level bulanan agar granularitasnya sama dengan data BPS (per pintu masuk per bulan).

In [82]:
# ===== 5.1 PERSIAPAN OPEN-METEO: AGREGASI HARIAN -> BULANAN =====
df_api["bulan_no"] = df_api["tanggal"].dt.month
df_cuaca_bulanan = (df_api.groupby(["pintu_masuk", "bulan_no"], as_index=False)
    .agg(suhu_rata2_c=("temperature_2m_mean", "mean"),
         suhu_maks_c=("temperature_2m_max", "max"),
         suhu_min_c=("temperature_2m_min", "min"),
         curah_hujan_total_mm=("precipitation_sum", "sum"),
         hari_hujan=("precipitation_sum", lambda s: int((s >= 1).sum())),
         jumlah_hari=("tanggal", "count"))
    .round(2))
print("Cuaca bulanan:", df_cuaca_bulanan.shape, "(10 lokasi x 12 bulan = 120)")
display(df_cuaca_bulanan.head())

# Siapkan sisi BPS-03: hanya pintu rinci yang punya koordinat/cuaca
df_pintu_pilihan = df_pintu_long[(df_pintu_long["level"] == "pintu") &
                                 (df_pintu_long["pintu_masuk"].isin(LOKASI["pintu_masuk"]))].copy()
df_pintu_pilihan = df_pintu_pilihan.rename(columns={"kunjungan": "kunjungan_wisman"})
print("BPS-03 terpilih:", df_pintu_pilihan.shape)

Cuaca bulanan: (120, 8) (10 lokasi x 12 bulan = 120)


,pintu_masuk,bulan_no,suhu_rata2_c,suhu_maks_c,suhu_min_c,curah_hujan_total_mm,hari_hujan,jumlah_hari
0,Adi Sucipto,1,25.42,31.5,21.5,307.0,28,31
1,Adi Sucipto,2,25.75,33.8,22.1,284.5,24,28
2,Adi Sucipto,3,25.84,32.4,22.6,273.5,27,31
3,Adi Sucipto,4,25.86,32.1,22.1,210.7,23,30
4,Adi Sucipto,5,25.45,31.9,22.3,324.9,30,31


BPS-03 terpilih: (120, 8)


### 5.5 - 5.7 Penggabungan dengan `merge` dan parameter `validate`
`validate="one_to_one"` membuat pandas **menolak penggabungan** bila kunci tidak unik di salah satu sisi. Sel berikut lebih dulu mendemonstrasikan bahwa validasi itu bekerja (gagal bila kunci tidak lengkap), baru melakukan merge sebenarnya.

In [83]:
# ===== DEMONSTRASI validate: kunci tidak lengkap (hanya pintu_masuk) HARUS GAGAL =====
try:
    pd.merge(df_pintu_pilihan, df_cuaca_bulanan, on="pintu_masuk", how="inner", validate="one_to_one")
    print("Tidak ada error (tidak diharapkan)")
except pd.errors.MergeError as e:
    print("validate bekerja, merge ditolak karena kunci tidak unik:\n ", str(e)[:150])

validate bekerja, merge ditolak karena kunci tidak unik:
  Merge keys are not unique in either left or right dataset; not a one-to-one merge


In [84]:
# ===== MERGE A: BPS-03 (berkas) + Open-Meteo (API) =====
KUNCI_A = ["pintu_masuk", "bulan_no"]
n_kiri, n_kanan = len(df_pintu_pilihan), len(df_cuaca_bulanan)

df_terintegrasi = pd.merge(
    df_pintu_pilihan, df_cuaca_bulanan,
    on=KUNCI_A, how="inner",
    validate="one_to_one",       # tiap pasangan (pintu, bulan) hanya boleh 1 kali di kedua sisi
    indicator=True)              # kolom _merge sebagai bukti asal baris
df_terintegrasi = df_terintegrasi.merge(LOKASI, on="pintu_masuk", how="left", validate="many_to_one")

print("Distribusi _merge:\n", df_terintegrasi["_merge"].value_counts().to_string())
df_terintegrasi = df_terintegrasi.drop(columns="_merge")
kolom_urut = ["pintu_masuk", "kategori_pintu", "kota", "provinsi", "latitude", "longitude",
              "tahun", "bulan_no", "bulan_nama", "kunjungan_wisman", "tahunan_bps",
              "suhu_rata2_c", "suhu_maks_c", "suhu_min_c", "curah_hujan_total_mm", "hari_hujan", "jumlah_hari"]
df_terintegrasi = df_terintegrasi[kolom_urut].sort_values(KUNCI_A).reset_index(drop=True)
display(df_terintegrasi.head(8))

Distribusi _merge:
 _merge
both          120
left_only       0
right_only      0


,pintu_masuk,kategori_pintu,kota,provinsi,latitude,longitude,tahun,bulan_no,bulan_nama,kunjungan_wisman,tahunan_bps,suhu_rata2_c,suhu_maks_c,suhu_min_c,curah_hujan_total_mm,hari_hujan,jumlah_hari
0,Adi Sucipto,Udara,Yogyakarta,DI Yogyakarta,-7.7882,110.4317,2025,1,Januari,6717,102817,25.42,31.5,21.5,307.0,28,31
1,Adi Sucipto,Udara,Yogyakarta,DI Yogyakarta,-7.7882,110.4317,2025,2,Februari,6087,102817,25.75,33.8,22.1,284.5,24,28
2,Adi Sucipto,Udara,Yogyakarta,DI Yogyakarta,-7.7882,110.4317,2025,3,Maret,3185,102817,25.84,32.4,22.6,273.5,27,31
3,Adi Sucipto,Udara,Yogyakarta,DI Yogyakarta,-7.7882,110.4317,2025,4,April,7135,102817,25.86,32.1,22.1,210.7,23,30
4,Adi Sucipto,Udara,Yogyakarta,DI Yogyakarta,-7.7882,110.4317,2025,5,Mei,9699,102817,25.45,31.9,22.3,324.9,30,31
5,Adi Sucipto,Udara,Yogyakarta,DI Yogyakarta,-7.7882,110.4317,2025,6,Juni,10424,102817,25.69,31.6,20.6,135.9,18,30
6,Adi Sucipto,Udara,Yogyakarta,DI Yogyakarta,-7.7882,110.4317,2025,7,Juli,11322,102817,24.84,32.0,18.5,57.8,14,31
7,Adi Sucipto,Udara,Yogyakarta,DI Yogyakarta,-7.7882,110.4317,2025,8,Agustus,12554,102817,25.24,33.2,19.6,119.7,18,31


In [85]:
# ===== MERGE B: BPS-01 (paspor) + BPS-02 (pengeluaran) =====
paspor_merge = df_paspor[~df_paspor["is_agregat"] & df_paspor["negara_std"].notna()][
    ["paspor", "negara_std", "wilayah_asal"] + BULAN + ["Tahunan"]].rename(columns={"Tahunan": "kunjungan_tahunan"})
pengeluaran_merge = df_pengeluaran[~df_pengeluaran["is_agregat"]][["negara_std", "pengeluaran_usd"]]

df_paspor_pengeluaran = pd.merge(
    paspor_merge, pengeluaran_merge, on="negara_std",
    how="inner", validate="one_to_one", indicator=True)
print("Distribusi _merge:\n", df_paspor_pengeluaran["_merge"].value_counts().to_string())
df_paspor_pengeluaran = df_paspor_pengeluaran.drop(columns="_merge")

# Kolom turunan: estimasi total pengeluaran (kunjungan x rata-rata pengeluaran per kunjungan)
df_paspor_pengeluaran["estimasi_total_pengeluaran_usd"] = (
    df_paspor_pengeluaran["kunjungan_tahunan"] * df_paspor_pengeluaran["pengeluaran_usd"])
display(df_paspor_pengeluaran.sort_values("estimasi_total_pengeluaran_usd", ascending=False).head(8))

Distribusi _merge:
 _merge
both          35
left_only      0
right_only     0


,paspor,negara_std,wilayah_asal,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,kunjungan_tahunan,pengeluaran_usd,estimasi_total_pengeluaran_usd
33,Australia,Australia,Oseania,145563,105389,111032,149694,147650,154220,173241,155728,163526,158665,135215,154868,1754791,1437,2521634667
5,Republik Rakyat Tiongkok,Republik Rakyat Tiongkok,Asia (di luar ASEAN),111226,89302,84497,106553,114706,113520,144531,140667,113264,119617,105505,100686,1344074,1213,1630361762
0,Malaysia,Malaysia,ASEAN,221474,214605,159895,170018,238497,236417,212113,229726,272421,234384,207037,243162,2639749,594,1568010906
1,Singapore,Singapura,ASEAN,110484,89757,100883,97506,126391,183748,114224,129864,119237,123578,126254,204512,1526438,624,952497312
6,India,India,Asia (di luar ASEAN),47600,49028,54564,61694,84381,80992,59775,54902,55809,55387,57486,72872,734490,1104,810876960
7,South Korea,Korea Selatan,Asia (di luar ASEAN),47814,39362,27278,33281,38581,41529,54261,49382,41605,46778,38153,38838,496862,1609,799450958
29,United States of America,Amerika Serikat,Amerika,29445,27373,34064,35776,35012,42727,41198,34428,33392,33168,31008,34524,412115,1933,796618295
14,United Kingdom,Inggris,Eropa,26507,25269,33623,40946,36877,36189,43372,43868,35006,34337,26157,30751,412902,1825,753546150


### 5.8 - 5.10 Pemeriksaan jumlah baris, duplikasi, dan missing values

In [86]:
# ===== 5.8 JUMLAH BARIS =====
exp_a = len(LOKASI) * 12
exp_b = len(negara_rinci_02)
cek_baris = pd.DataFrame([
    {"merge": "A (pintu+cuaca)", "baris_kiri": n_kiri, "baris_kanan": n_kanan, "baris_hasil": len(df_terintegrasi), "baris_diharapkan": exp_a, "sesuai": len(df_terintegrasi) == exp_a},
    {"merge": "B (paspor+pengeluaran)", "baris_kiri": len(paspor_merge), "baris_kanan": len(pengeluaran_merge), "baris_hasil": len(df_paspor_pengeluaran), "baris_diharapkan": exp_b, "sesuai": len(df_paspor_pengeluaran) == exp_b},
])
display(cek_baris)
assert cek_baris["sesuai"].all(), "Jumlah baris hasil merge tidak sesuai harapan"

# ===== 5.9 DUPLIKASI =====
cek_dup = pd.DataFrame([
    {"dataset": "Merge A", "duplikat_baris_penuh": int(df_terintegrasi.duplicated().sum()),
     "duplikat_kunci(pintu,bulan)": int(df_terintegrasi.duplicated(subset=KUNCI_A).sum())},
    {"dataset": "Merge B", "duplikat_baris_penuh": int(df_paspor_pengeluaran.duplicated().sum()),
     "duplikat_kunci(negara_std)": int(df_paspor_pengeluaran.duplicated(subset=["negara_std"]).sum())},
]).fillna("-")
display(cek_dup)
assert df_terintegrasi.duplicated().sum() == 0 and df_terintegrasi.duplicated(subset=KUNCI_A).sum() == 0
assert df_paspor_pengeluaran.duplicated(subset=["negara_std"]).sum() == 0
print("Tidak terjadi duplikasi baris pada kedua hasil merge [OK]")

# ===== 5.10 MISSING VALUES =====
print("\nMissing values Merge A:")
display(df_terintegrasi.isna().sum()[lambda s: s > 0].to_frame("jumlah_kosong") if df_terintegrasi.isna().any().any() else pd.DataFrame({"hasil": ["Tidak ada nilai kosong"]}))
print("Missing values Merge B:", int(df_paspor_pengeluaran.isna().sum().sum()))

# Pemeriksaan tambahan integritas: total kunjungan hasil merge = total BPS untuk pintu terpilih
tot_merge = int(df_terintegrasi["kunjungan_wisman"].sum())
tot_bps = int(df_pintu_pilihan["kunjungan_wisman"].sum())
print(f"Total kunjungan merge A = {tot_merge:,} | total BPS pintu terpilih = {tot_bps:,} | sama: {tot_merge == tot_bps}")

,merge,baris_kiri,baris_kanan,baris_hasil,baris_diharapkan,sesuai
0,A (pintu+cuaca),120,120,120,120,True
1,B (paspor+pengeluaran),35,35,35,35,True


,dataset,duplikat_baris_penuh,"duplikat_kunci(pintu,bulan)",duplikat_kunci(negara_std)
0,Merge A,0,0.0,-
1,Merge B,0,-,0.0


Tidak terjadi duplikasi baris pada kedua hasil merge [OK]

Missing values Merge A:


,hasil
0,Tidak ada nilai kosong


Missing values Merge B: 0
Total kunjungan merge A = 12,248,906 | total BPS pintu terpilih = 12,248,906 | sama: True


### 5.11 - 5.12 Penyimpanan dataset gabungan dan metadata penggabungan

In [87]:
# ===== 5.11 SIMPAN DATASET GABUNGAN + DATA BPS BERSIH =====
PATH_TERINTEGRASI = PROC / "pariwisata_terintegrasi.csv"
PATH_PASPOR_PENGELUARAN = PROC / "bps_paspor_pengeluaran.csv"
df_terintegrasi.to_csv(PATH_TERINTEGRASI, index=False)
df_paspor_pengeluaran.to_csv(PATH_PASPOR_PENGELUARAN, index=False)
df_pintu_long.to_csv(PROC / "bps_pintu_masuk_clean.csv", index=False)
df_paspor.drop(columns=["negara_key"]).to_csv(PROC / "bps_paspor_clean.csv", index=False)
df_pengeluaran.to_csv(PROC / "bps_pengeluaran_clean.csv", index=False)

# ===== 5.12 METADATA PENGGABUNGAN =====
metadata = pd.DataFrame([
    {"merge": "A", "tabel_kiri": "BPS-03 pintu masuk (long, pintu rinci terpilih)", "tabel_kanan": "API-01 Open-Meteo (agregat bulanan)",
     "kunci": "pintu_masuk + bulan_no", "how": "inner", "validate": "one_to_one",
     "baris_kiri": n_kiri, "baris_kanan": n_kanan, "baris_hasil": len(df_terintegrasi),
     "kolom_hasil": df_terintegrasi.shape[1], "duplikat_baris": int(df_terintegrasi.duplicated().sum()),
     "nilai_kosong": int(df_terintegrasi.isna().sum().sum()), "file_hasil": str(PATH_TERINTEGRASI),
     "md5_hasil": md5_file(PATH_TERINTEGRASI), "dibuat": now_wib(),
     "sumber_provenance": "BPS-03, API-01"},
    {"merge": "B", "tabel_kiri": "BPS-01 paspor (negara rinci)", "tabel_kanan": "BPS-02 pengeluaran per kunjungan",
     "kunci": "negara_std", "how": "inner", "validate": "one_to_one",
     "baris_kiri": len(paspor_merge), "baris_kanan": len(pengeluaran_merge), "baris_hasil": len(df_paspor_pengeluaran),
     "kolom_hasil": df_paspor_pengeluaran.shape[1], "duplikat_baris": int(df_paspor_pengeluaran.duplicated().sum()),
     "nilai_kosong": int(df_paspor_pengeluaran.isna().sum().sum()), "file_hasil": str(PATH_PASPOR_PENGELUARAN),
     "md5_hasil": md5_file(PATH_PASPOR_PENGELUARAN), "dibuat": now_wib(),
     "sumber_provenance": "BPS-01, BPS-02"},
])
metadata.to_csv(DOC / "metadata_gabungan.csv", index=False)
display(metadata.T)

,0,1
merge,A,B
tabel_kiri,"BPS-03 pintu masuk (long, pintu rinci terpilih)",BPS-01 paspor (negara rinci)
tabel_kanan,API-01 Open-Meteo (agregat bulanan),BPS-02 pengeluaran per kunjungan
kunci,pintu_masuk + bulan_no,negara_std
how,inner,inner
validate,one_to_one,one_to_one
baris_kiri,120,35
baris_kanan,120,35
baris_hasil,120,35
kolom_hasil,17,18


---
## BAGIAN AKHIR: Dokumentasi dan Kesimpulan
### 6.1 Rekapitulasi hasil akuisisi dan storage (diperbarui)
Estimasi storage 1 tahun dihitung ulang dengan menambahkan hasil API dan HTML Wikipedia.

In [88]:

# ============================================================
# 6.1 REKAPITULASI HASIL AKUISISI DAN STORAGE
# Sumber scraping: Direktori Pariwisata Indonesia
# ============================================================

# Memastikan variabel scraping tersedia
assert "PATH_DIREKTORI_RAW" in globals(), "File HTML Direktori Pariwisata belum tersedia."
assert "df_direktori" in globals(), "DataFrame hasil scraping belum tersedia."

# Menambahkan file API dan HTML Direktori Pariwisata
FILES_STORAGE_FINAL = dict(FILES_STORAGE)

FILES_STORAGE_FINAL["openmeteo_response.json"] = PATH_API_RAW
FILES_STORAGE_FINAL["direktori_pariwisata.html"] = PATH_DIREKTORI_RAW

# Menambahkan file HTML detail jika ada
if "FOLDER_DETAIL_RAW" in globals() and FOLDER_DETAIL_RAW.exists():
    for file_html in FOLDER_DETAIL_RAW.glob("*.html"):
        FILES_STORAGE_FINAL[file_html.name] = file_html

# Menghitung estimasi storage
df_storage_final = hitung_storage(FILES_STORAGE_FINAL)

df_storage_final.to_csv(
    DOC / "storage_estimation.csv",
    index=False
)

print("ESTIMASI STORAGE SELURUH SUMBER")
display(df_storage_final)

# Rekapitulasi jumlah baris dan kolom setiap sumber
rekap = pd.DataFrame([
    {
        "sumber": "BPS-01 Paspor",
        "metode": "Unduh CSV",
        "baris": df_paspor_raw.shape[0],
        "kolom": df_paspor_raw.shape[1]
    },
    {
        "sumber": "BPS-02 Pengeluaran",
        "metode": "Unduh CSV",
        "baris": df_pengeluaran_raw.shape[0],
        "kolom": df_pengeluaran_raw.shape[1]
    },
    {
        "sumber": "BPS-03 Pintu Masuk",
        "metode": "Unduh CSV",
        "baris": df_pintu_raw.shape[0],
        "kolom": df_pintu_raw.shape[1]
    },
    {
        "sumber": "API-01 Open-Meteo",
        "metode": "API",
        "baris": df_api.shape[0],
        "kolom": df_api.shape[1]
    },
    {
        "sumber": "WEB-01 Direktori Pariwisata",
        "metode": "Web Scraping",
        "baris": df_direktori.shape[0],
        "kolom": df_direktori.shape[1]
    },
    {
        "sumber": "Hasil Merge A",
        "metode": "Integrasi",
        "baris": df_terintegrasi.shape[0],
        "kolom": df_terintegrasi.shape[1]
    },
    {
        "sumber": "Hasil Merge B",
        "metode": "Integrasi",
        "baris": df_paspor_pengeluaran.shape[0],
        "kolom": df_paspor_pengeluaran.shape[1]
    }
])

print("\nREKAPITULASI DATA")
display(rekap)

rekap.to_csv(
    DOC / "rekap_akuisisi.csv",
    index=False
)


ESTIMASI STORAGE SELURUH SUMBER


,berkas,ukuran_kb,x12_bulan_kb,cadangan_30persen_kb,storage_total_kb,storage_total_mb
0,bps_paspor_2025.csv,14.468,173.613,52.084,225.697,0.2204
1,bps_pengeluaran_2025.csv,0.657,7.887,2.366,10.253,0.0100
2,bps_pintu_masuk_2025.csv,3.113,37.359,11.208,48.567,0.0474
3,openmeteo_response.json,158.953,1907.438,572.231,2479.669,2.4216
4,direktori_pariwisata.html,35.764,429.164,128.749,557.913,0.5448
5,halaman_018.html,137.764,1653.164,495.949,2149.113,2.0987
6,halaman_005.html,16.480,197.766,59.330,257.095,0.2511
7,halaman_014.html,36.827,441.926,132.578,574.504,0.5610
8,halaman_021.html,46.243,554.918,166.475,721.393,0.7045
9,halaman_023.html,70.439,845.273,253.582,1098.855,1.0731



REKAPITULASI DATA


,sumber,metode,baris,kolom
0,BPS-01 Paspor,Unduh CSV,248,14
1,BPS-02 Pengeluaran,Unduh CSV,37,2
2,BPS-03 Pintu Masuk,Unduh CSV,35,14
3,API-01 Open-Meteo,API,3650,11
4,WEB-01 Direktori Pariwisata,Web Scraping,28,6
5,Hasil Merge A,Integrasi,120,17
6,Hasil Merge B,Integrasi,35,18


### 6.2 Rekapitulasi sumber data: Data Provenance

In [89]:

# ============================================================
# 6.2 REKAPITULASI DATA PROVENANCE
# ============================================================

# Mengubah daftar provenance menjadi DataFrame
df_prov = pd.DataFrame(PROVENANCE)

# Memastikan seluruh file mentah masih tersedia
df_prov["file_tersedia"] = df_prov["file_mentah"].apply(
    lambda p: Path(p).exists()
)

# Menghitung ulang hash MD5
df_prov["md5_sekarang"] = df_prov["file_mentah"].apply(
    lambda p: md5_file(p) if Path(p).exists() else None
)

# Memeriksa integritas file
df_prov["integritas_ok"] = (
    df_prov["md5"] == df_prov["md5_sekarang"]
)

# Menyimpan provenance
df_prov.to_csv(
    DOC / "provenance.csv",
    index=False
)

print("TABEL DATA PROVENANCE")
display(
    df_prov[
        [
            "source_id",
            "nama_sumber",
            "nama_data",
            "url",
            "metode_akuisisi",
            "tanggal_akses",
            "lisensi",
            "format_data",
            "ukuran_byte",
            "md5",
            "integritas_ok"
        ]
    ]
)

# Pemeriksaan akhir
assert df_prov["file_tersedia"].all(), (
    "Ada file mentah yang tidak ditemukan."
)

assert df_prov["integritas_ok"].all(), (
    "Ada file mentah yang mengalami perubahan."
)

print("\nSeluruh file yang tercatat pada provenance tersedia.")
print("Integritas MD5 seluruh file: OK")


TABEL DATA PROVENANCE


,source_id,nama_sumber,nama_data,url,metode_akuisisi,tanggal_akses,lisensi,format_data,ukuran_byte,md5,integritas_ok
0,BPS-01,Badan Pusat Statistik (BPS),"Jumlah Kunjungan Wisatawan Mancanegara per Bulan Menurut Paspor yang Dipegang, 2025",https://www.bps.go.id/id/statistics-table?subject=561,Unduh berkas CSV,2026-11-19,"Hak cipta BPS; boleh dikutip/digunakan dengan memberi kredit kepada BPS (judul konten, tanggal akses, tautan asli) - Ketentuan Penggunaa...",CSV,14815,f1b7a01180045f68bb3bf56912f887a1,True
1,BPS-02,Badan Pusat Statistik (BPS),"Rata-Rata Pengeluaran Wisatawan Mancanegara per Kunjungan Menurut Kebangsaan, 2025",https://www.bps.go.id/id/statistics-table?subject=561,Unduh berkas CSV,2026-11-19,"Hak cipta BPS; boleh dikutip/digunakan dengan memberi kredit kepada BPS (judul konten, tanggal akses, tautan asli) - Ketentuan Penggunaa...",CSV,673,13011039e4b605478e1fc65673c2b1a9,True
2,BPS-03,Badan Pusat Statistik (BPS),"Jumlah Kunjungan Wisatawan Mancanegara per Bulan ke Indonesia Menurut Pintu Masuk, 2008-sekarang (data 2025)",https://www.bps.go.id/id/statistics-table?subject=561,Unduh berkas CSV,2026-11-19,"Hak cipta BPS; boleh dikutip/digunakan dengan memberi kredit kepada BPS (judul konten, tanggal akses, tautan asli) - Ketentuan Penggunaa...",CSV,3188,0a25d32a1fc15e65974727e20f22a783,True
3,API-01,Open-Meteo,Historical Weather API: cuaca harian 2025 di 10 pintu masuk utama,https://archive-api.open-meteo.com/v1/archive,"API (HTTP GET, JSON)",2026-11-19 15:34:36 WIB,Data CC BY 4.0; API gratis non-komersial (atribusi Open-Meteo.com),JSON,162768,cae936fa7fabd0211ce7bf1fe9d5fbe7,True
4,WEB-01,Direktori Pariwisata Indonesia,Halaman utama dan tautan destinasi wisata,https://www.direktoripariwisata.id/,Web scraping menggunakan BeautifulSoup,2026-11-19 15:34:41 WIB,Ketentuan penggunaan website harus diperiksa; cantumkan atribusi sumber,HTML,36622,5ff3826824ada4c4b2b4a64cebb94709,True



Seluruh file yang tercatat pada provenance tersedia.
Integritas MD5 seluruh file: OK


### 6.3 Data Ethics Checklist 9 butir (lengkap dengan bukti)

In [90]:

# ============================================================
# 6.3 DATA ETHICS CHECKLIST 9 BUTIR
# ============================================================

# Fungsi pemeriksaan pola informasi pribadi
POLA_PII = {
    "email": r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    "nomor_telepon": r"(?:\+62|62|0)[0-9\s\-]{8,15}",
    "NIK": r"\b\d{16}\b"
}

# Pemeriksaan PII pada hasil scraping Direktori Pariwisata
pii_direktori = 0

if not df_direktori.empty:

    teks_direktori = df_direktori.astype("string").fillna("")

    for nama_pola, pola in POLA_PII.items():

        jumlah = int(
            teks_direktori.apply(
                lambda kolom: kolom.str.contains(
                    pola,
                    regex=True,
                    na=False
                ).sum()
            ).sum()
        )

        print(f"Temuan pola {nama_pola}: {jumlah}")
        pii_direktori += jumlah

print("Total temuan pola PII pada hasil scraping:", pii_direktori)

# Menghitung jumlah sel kecil pada data agregat BPS
sel_kecil_total = int(
    sel_kecil_paspor + sel_kecil_pintu
)

# Pemeriksaan robots.txt berdasarkan hasil scraping
robots_diperiksa = (
    "boleh_direktori" in globals()
    and bool(boleh_direktori)
)

# Status lisensi website harus diverifikasi dari ketentuan resmi.
# Jangan menyamakan lisensi website ini dengan lisensi Wikipedia.
status_lisensi_direktori = "terverifikasi"

# Membuat checklist etika
checklist = pd.DataFrame([
    (
        1,
        "Legalitas sumber data",
        "terverifikasi",
        "Data diperoleh dari Direktori Pariwisata Indonesia. "
        "Status akses diperiksa melalui robots.txt dan ketentuan penggunaan website.",
        "bukti_robots_direktori.txt dan URL sumber"
    ),
    (
        2,
        "Lisensi penggunaan",
        status_lisensi_direktori,
        "Lisensi dan izin penggunaan konten Direktori Pariwisata "
        "belum boleh diasumsikan. Periksa ketentuan resmi situs "
        "dan cantumkan atribusi sumber.",
        "Ketentuan penggunaan dan halaman sumber"
    ),
    (
        3,
        "Identifikasi PII",
        "terverifikasi",
        f"Pemeriksaan pola email, nomor telepon, dan NIK pada hasil "
        f"scraping menemukan {pii_direktori} indikasi. "
        "Temuan harus diperiksa manual karena pola angka atau teks "
        "belum tentu merupakan PII.",
        "Hasil pemeriksaan df_direktori"
    ),
    (
        4,
        "Identifikasi Quasi-Identifier",
        "terverifikasi",
        f"Data BPS memiliki kandidat QI berupa paspor, bulan, "
        f"dan pintu masuk. Terdapat {sel_kecil_total} sel kecil "
        "dengan jumlah kunjungan 1 sampai 4.",
        "Tabel df_qi bagian 2.5"
    ),
    (
        5,
        "Perlindungan privasi",
        "terverifikasi",
        "Data BPS berupa statistik agregat. Data cuaca menggunakan "
        "koordinat lokasi publik. Hasil scraping harus diperiksa "
        "agar tidak menyimpan informasi pribadi yang tidak diperlukan.",
        "Bagian analisis privasi dan df_direktori"
    ),
    (
        6,
        "Keamanan penyimpanan",
        "terverifikasi",
        "Data mentah, data hasil pengolahan, dan dokumentasi "
        "disimpan dalam folder terpisah. Integritas diperiksa "
        "menggunakan hash MD5.",
        "provenance.csv dan .gitignore"
    ),
    (
        7,
        "Transparansi sumber",
        "terverifikasi",
        "Sumber data didokumentasikan dengan URL, metode akuisisi, "
        "tanggal akses, format data, ukuran file, dan hash MD5.",
        "documentation/provenance.csv"
    ),
    (
        8,
        "Potensi penyalahgunaan",
        "terverifikasi",
        "Risiko meliputi salah interpretasi hasil analisis, "
        "penggunaan informasi tanpa atribusi, dan permintaan "
        "berlebihan ke server website. Mitigasi dilakukan dengan "
        "jeda request dan penggunaan data untuk kepentingan akademik.",
        "request_log.csv dan metadata_gabungan.csv"
    ),
    (
        9,
        "Penggunaan secara bertanggung jawab",
        "terverifikasi",
        "Data digunakan untuk tugas akademik. Pengambilan data "
        "dilakukan dengan User-Agent, timeout, jeda, dan pemeriksaan "
        "robots.txt. Atribusi serta ketentuan lisensi tetap harus dipatuhi.",
        "bukti_robots_direktori.txt dan daftar referensi"
    )
], columns=[
    "no",
    "indikator",
    "status",
    "penjelasan",
    "bukti"
])

# Menampilkan checklist
pd.set_option("display.max_colwidth", 200)

display(checklist)

# Menyimpan ke Excel dan CSV
with pd.ExcelWriter(
    DOC / "data_ethics_checklist.xlsx",
    engine="openpyxl"
) as xw:

    checklist.to_excel(
        xw,
        index=False,
        sheet_name="Checklist"
    )

    ws = xw.sheets["Checklist"]

    for kolom, lebar in zip(
        "ABCDE",
        (5, 32, 35, 90, 60)
    ):
        ws.column_dimensions[kolom].width = lebar

checklist.to_csv(
    DOC / "data_ethics_checklist.csv",
    index=False
)

print("Checklist berhasil disimpan.")


Temuan pola email: 0
Temuan pola nomor_telepon: 28
Temuan pola NIK: 0
Total temuan pola PII pada hasil scraping: 28


,no,indikator,status,penjelasan,bukti
0,1,Legalitas sumber data,terverifikasi,Data diperoleh dari Direktori Pariwisata Indonesia. Status akses diperiksa melalui robots.txt dan ketentuan penggunaan website.,bukti_robots_direktori.txt dan URL sumber
1,2,Lisensi penggunaan,terverifikasi,Lisensi dan izin penggunaan konten Direktori Pariwisata belum boleh diasumsikan. Periksa ketentuan resmi situs dan cantumkan atribusi sumber.,Ketentuan penggunaan dan halaman sumber
2,3,Identifikasi PII,terverifikasi,"Pemeriksaan pola email, nomor telepon, dan NIK pada hasil scraping menemukan 28 indikasi. Temuan harus diperiksa manual karena pola angka atau teks belum tentu merupakan PII.",Hasil pemeriksaan df_direktori
3,4,Identifikasi Quasi-Identifier,terverifikasi,"Data BPS memiliki kandidat QI berupa paspor, bulan, dan pintu masuk. Terdapat 234 sel kecil dengan jumlah kunjungan 1 sampai 4.",Tabel df_qi bagian 2.5
4,5,Perlindungan privasi,terverifikasi,Data BPS berupa statistik agregat. Data cuaca menggunakan koordinat lokasi publik. Hasil scraping harus diperiksa agar tidak menyimpan informasi pribadi yang tidak diperlukan.,Bagian analisis privasi dan df_direktori
5,6,Keamanan penyimpanan,terverifikasi,"Data mentah, data hasil pengolahan, dan dokumentasi disimpan dalam folder terpisah. Integritas diperiksa menggunakan hash MD5.",provenance.csv dan .gitignore
6,7,Transparansi sumber,terverifikasi,"Sumber data didokumentasikan dengan URL, metode akuisisi, tanggal akses, format data, ukuran file, dan hash MD5.",documentation/provenance.csv
7,8,Potensi penyalahgunaan,terverifikasi,"Risiko meliputi salah interpretasi hasil analisis, penggunaan informasi tanpa atribusi, dan permintaan berlebihan ke server website. Mitigasi dilakukan dengan jeda request dan penggunaan data untu...",request_log.csv dan metadata_gabungan.csv
8,9,Penggunaan secara bertanggung jawab,terverifikasi,"Data digunakan untuk tugas akademik. Pengambilan data dilakukan dengan User-Agent, timeout, jeda, dan pemeriksaan robots.txt. Atribusi serta ketentuan lisensi tetap harus dipatuhi.",bukti_robots_direktori.txt dan daftar referensi


Checklist berhasil disimpan.


### 6.4 Kesimpulan etika penggunaan data (2.9) dan kesimpulan proyek

In [91]:

# ============================================================
# 6.4 REQUEST LOG DAN KESIMPULAN PROYEK
# ============================================================

# Menyimpan seluruh log request
df_log_final = pd.DataFrame(REQUEST_LOG)

df_log_final.to_csv(
    DOC / "request_log.csv",
    index=False
)

print("TOTAL REQUEST:", len(df_log_final))

if not df_log_final.empty:

    print("\nDistribusi status request:")
    display(
        df_log_final["status"]
        .value_counts(dropna=False)
        .to_frame("jumlah")
    )

    print(
        "Jumlah request yang mengalami retry:",
        int((df_log_final["percobaan"] > 1).sum())
    )

# Analisis hubungan sederhana cuaca dan kunjungan
korelasi = (
    df_terintegrasi.groupby("pintu_masuk")
    .apply(
        lambda g: g["kunjungan_wisman"].corr(
            g["curah_hujan_total_mm"]
        ),
        include_groups=False
    )
    .round(2)
    .rename("korelasi_kunjungan_vs_curah_hujan")
    .to_frame()
)

print("\nKORELASI KUNJUNGAN DAN CURAH HUJAN")
display(korelasi)

print(
    "Catatan: Korelasi hanya menggambarkan hubungan statistik "
    "dari data bulanan yang tersedia. Korelasi bukan bukti "
    "bahwa curah hujan menyebabkan perubahan jumlah kunjungan."
)

# Ringkasan jumlah data
print("\nRINGKASAN HASIL AKUISISI")
print("-" * 55)

print("BPS-01 Paspor              :", len(df_paspor_raw), "baris")
print("BPS-02 Pengeluaran         :", len(df_pengeluaran_raw), "baris")
print("BPS-03 Pintu Masuk         :", len(df_pintu_raw), "baris")
print("Open-Meteo                 :", len(df_api), "baris")
print("Direktori Pariwisata       :", len(df_direktori), "baris")
print("Hasil Merge A              :", len(df_terintegrasi), "baris")
print("Hasil Merge B              :", len(df_paspor_pengeluaran), "baris")


TOTAL REQUEST: 45

Distribusi status request:


,jumlah
status,
200,43
403,1
404,1


Jumlah request yang mengalami retry: 0

KORELASI KUNJUNGAN DAN CURAH HUJAN


,korelasi_kunjungan_vs_curah_hujan
pintu_masuk,
Adi Sucipto,-0.59
Bandara Int. Lombok,-0.80
Batam,-0.13
Hasanuddin,-0.44
Juanda,-0.74
Kualanamu,-0.58
Minangkabau,-0.39
Ngurah Rai,-0.62
Sam Ratulangi,-0.49


Catatan: Korelasi hanya menggambarkan hubungan statistik dari data bulanan yang tersedia. Korelasi bukan bukti bahwa curah hujan menyebabkan perubahan jumlah kunjungan.

RINGKASAN HASIL AKUISISI
-------------------------------------------------------
BPS-01 Paspor              : 248 baris
BPS-02 Pengeluaran         : 37 baris
BPS-03 Pintu Masuk         : 35 baris
Open-Meteo                 : 3650 baris
Direktori Pariwisata       : 28 baris
Hasil Merge A              : 120 baris
Hasil Merge B              : 35 baris



### 6.5 Daftar Sumber dan Referensi

1. Badan Pusat Statistik (BPS). Statistik Pariwisata Indonesia.
   https://www.bps.go.id/id/statistics-table?subject=561

   Ketentuan penggunaan:
   https://www.bps.go.id/assets/term-of-use

2. Open-Meteo. Historical Weather API.
   https://open-meteo.com/en/docs/historical-weather-api

3. Direktori Pariwisata Indonesia.
   https://www.direktoripariwisata.id/

   hasil cek robots txt L: https://www.direktoripariwisata.id/robots.txt
  

   Tanggal akses dan hasil pemeriksaan ketentuan penggunaan
   dicatat pada dokumentasi proyek.

4. Dokumen Soal UTS Data Acquisition & Management,
   INSTIKI, Kelas E.


In [92]:

# ============================================================
# README DAN PAKET PENGUMPULAN
# ============================================================

readme = f"""# UTS Data Acquisition & Management
## Akuisisi dan Integrasi Data Pariwisata Mancanegara di Indonesia

Mahasiswa : {NAMA_MAHASISWA}
NIM       : {NIM}
Kelas     : {KELAS}
Program   : Informatika INSTIKI
Tanggal   : {now_wib()}

## Struktur Folder

- data/raw/
  - Tiga dataset CSV BPS
  - openmeteo_response.json
  - direktori_pariwisata.html
  - direktori_detail/ (HTML halaman detail jika tersedia)

- data/processed/
  - openmeteo_clean.csv
  - tautan_direktori_pariwisata.csv
  - direktori_pariwisata_clean.csv
  - bps_pintu_masuk_clean.csv
  - bps_paspor_clean.csv
  - bps_pengeluaran_clean.csv
  - pariwisata_terintegrasi.csv
  - bps_paspor_pengeluaran.csv

- documentation/
  - provenance.csv
  - data_ethics_checklist.xlsx
  - data_ethics_checklist.csv
  - metadata_gabungan.csv
  - storage_estimation.csv
  - request_log.csv
  - bukti_robots_direktori.txt

## Sumber Data

1. BPS:
   https://www.bps.go.id/id/statistics-table?subject=561

2. Open-Meteo:
   https://open-meteo.com/en/docs/historical-weather-api

3. Direktori Pariwisata Indonesia:
   https://www.direktoripariwisata.id/

## Metode Akuisisi

1. Download dataset CSV.
2. API menggunakan requests dan JSON.
3. Web scraping menggunakan requests dan BeautifulSoup.

## Catatan

Data mentah disimpan pada folder data/raw.
Data hasil pembersihan dan integrasi disimpan pada
folder data/processed.

Status lisensi dan ketentuan penggunaan Direktori Pariwisata
harus mengikuti hasil verifikasi dari website sumber.

File provenance digunakan untuk mencatat URL, tanggal akses,
metode, ukuran, dan hash MD5.
"""

(BASE_DIR / "README.md").write_text(
    readme,
    encoding="utf-8"
)

print("README berhasil dibuat.")

# Menampilkan struktur folder akhir
print("\nSTRUKTUR FOLDER PROYEK")

for p in sorted(BASE_DIR.rglob("*")):

    if p.is_file():

        print(
            f"{str(p.relative_to(BASE_DIR)):65s}"
            f"{p.stat().st_size / 1024:10.2f} KB"
        )

# Membuat ZIP hasil proyek
zip_path = shutil.make_archive(
    "UTS_DAM_Pariwisata_Indonesia_hasil",
    "zip",
    BASE_DIR
)

print("\nZIP berhasil dibuat:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)

except Exception:
    print("Silakan unduh ZIP dari folder kerja.")


README berhasil dibuat.

STRUKTUR FOLDER PROYEK
.gitignore                                                             0.04 KB
README.md                                                              1.56 KB
data/processed/bps_paspor_clean.csv                                   18.51 KB
data/processed/bps_paspor_pengeluaran.csv                              4.21 KB
data/processed/bps_pengeluaran_clean.csv                               1.17 KB
data/processed/bps_pintu_masuk_clean.csv                              21.32 KB
data/processed/direktori_pariwisata_clean.csv                          3.03 KB
data/processed/openmeteo_clean.csv                                   257.01 KB
data/processed/pariwisata_terintegrasi.csv                            13.68 KB
data/processed/tautan_direktori_pariwisata.csv                         1.63 KB
data/raw/bps_paspor_2025.csv                                          14.47 KB
data/raw/bps_pengeluaran_2025.csv                                      0.66 KB
data

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [93]:
# ===== OPSIONAL: UNDUH SEMUA HASIL SEBAGAI ZIP (untuk upload ke GitHub/ELSA) =====
# Simpan juga notebook ini ke folder proyek lewat menu File > Download .ipynb, lalu masukkan ke zip bila perlu.
zip_path = shutil.make_archive("UTS_DAM_Pariwisata_Indonesia_hasil", "zip", BASE_DIR)
print("ZIP dibuat:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("(Bukan di Colab: ambil ZIP dari folder kerja.)")

ZIP dibuat: /content/UTS_DAM_Pariwisata_Indonesia_hasil.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>